# Michigan Medicine HITS — Incident Analysis Pipeline

Three-stage pipeline:
- **Stage 1 (Chunk A):** Baseline demand, weekly aggregation, workflow noise
- **Stage 2 (Chunk B):** Persistent burden (static) + episodic disruptions (temporal) → candidate list
- **Stage 3 (Chunk C):** Conditional NLP clustering, gated by Stage 2 candidates

## A0 — Setup & Load

In [2]:
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ── Pipeline configuration ────────────────────────────────────────────────────
# Single source of truth for all thresholds and parameters.
# Chunk B and C will import or copy this dict.

CONFIG = {
    # ── A1: Temporal cleaning ─────────────────────────────────────────────
    "timestamp_cols":       ["Created", "Resolved", "Closed"],

    # ── A3: Weekly aggregation ────────────────────────────────────────────
    "top_n_services":       60,          # services included in service_weekly

    # ── A4: Caller / workflow noise ───────────────────────────────────────
    "min_service_tickets":  50,          # minimum tickets for caller analysis
    "duplicate_gap_hours":  72,          # max gap between tickets in a cluster
    "automated_caller_re":  r"\b(api|autogen|bot|automated)\b",
    "generic_callers":      ["External Caller", "Guest", "Unknown",
                             "System", "Service Desk"],

    # ── B2: Spike detection (defined here, consumed in Chunk B) ───────────
    "baseline_weeks":       8,           # rolling window for baseline
    "min_baseline_periods": 4,           # minimum history before flagging
    "vol_ratio_threshold":  1.50,        # ≥50% above rolling median
    "min_vol_rel":          0.15,        # excess ≥ 15% of baseline
    "min_vol_abs":          40,          # hard floor on excess tickets
    "min_weekly_tickets":   20,          # exclude near-empty weeks
    "mttr_ratio_threshold": 1.50,
    "min_mttr_abs":         10.0,        # minimum meaningful MTTR excess (hours)
    "automated_mttr_ceil":  0.5,         # MTTR < this → automated batch

    # ── B3: Episode merging ───────────────────────────────────────────────
    "episode_max_gap":      1,           # non-spike weeks allowed in episode

    # ── B4: Candidate ranking ─────────────────────────────────────────────
    "top_k_episodes":       15,          # top-K episodes per type

    # ── C2: NLP gating ────────────────────────────────────────────────────
    "min_nlp_chars":        15,          # minimum cleaned text length
    "nlp_min_tickets":      100,         # minimum usable tickets to run NLP
    "nlp_max_tickets":      2000,        # subsample cap for large populations
    "nlp_top_k_episodic":   10,          # only run NLP on top-K episodic by severity
    "nlp_top_k_persistent": 10,          # only run NLP on top-K persistent by impact
    "dup_deflation_warn":   0.40,        # warn if >40% of candidate is duplicates
}

In [4]:
# ── Data load ─────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")

file_paths = sorted(DATA_DIR.glob("*.xlsx"))
print(f"Data files found: {file_paths}")

dfs = []
for fp in file_paths:
    sheets = pd.read_excel(fp, sheet_name=None)
    dfs.extend(sheets.values())

df = pd.concat(dfs, ignore_index=True)
print(f"Raw data: {df.shape[0]:,} rows × {df.shape[1]} columns")

Data files found: [PosixPath('data/2025-01 to 2025-03 Incidents.xlsx'), PosixPath('data/2025-04 to 2025-06 Incidents.xlsx'), PosixPath('data/2025-07 to 2025-09 Incidents.xlsx'), PosixPath('data/2025-10 to 2025-12 Incidents.xlsx')]


/Users/muyu/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/muyu/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/muyu/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Raw data: 274,560 rows × 25 columns


## A1 — Deduplication & Temporal Cleaning

One row per incident. All temporal columns derived here.
`df_clean` is the frozen output — no downstream stage mutates it (except the final A4 flags).

In [5]:
# ── Deduplication ─────────────────────────────────────────────────────────────
n_raw = len(df)
n_unique = df["Number"].nunique()
print(f"Raw rows: {n_raw:,}  |  Unique incidents: {n_unique:,}  |  Duplicates: {n_raw - n_unique:,}")

df_clean = df.drop_duplicates(subset="Number", keep="last").copy()
print(f"After dedup: {len(df_clean):,} rows")

Raw rows: 274,560  |  Unique incidents: 274,550  |  Duplicates: 10
After dedup: 274,550 rows


In [6]:
# ── Timestamp parsing ─────────────────────────────────────────────────────────
for col in CONFIG["timestamp_cols"]:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

print(f"Date range: {df_clean['Created'].min().date()} → {df_clean['Created'].max().date()}")

# ── Derived time metrics ──────────────────────────────────────────────────────
df_clean["MTTR_hours"]    = (df_clean["Resolved"] - df_clean["Created"]).dt.total_seconds() / 3600
df_clean["Time_to_close"] = (df_clean["Closed"]   - df_clean["Created"]).dt.total_seconds() / 3600
df_clean["closure_lag"]   = df_clean["Time_to_close"] - df_clean["MTTR_hours"]

# ── Temporal convenience columns ──────────────────────────────────────────────
df_clean["date"] = df_clean["Created"].dt.date
df_clean["hour"] = df_clean["Created"].dt.hour
df_clean["dow"]  = df_clean["Created"].dt.day_name()
df_clean["week"] = df_clean["Created"].dt.to_period("W").dt.start_time

# ── Trim partial boundary weeks ──────────────────────────────────────────────
# First and last weeks are likely incomplete; drop them to avoid false signals.
week_counts = df_clean["week"].value_counts().sort_index()
full_weeks  = week_counts.index[1:-1]  # drop first and last
n_before    = len(df_clean)
df_clean    = df_clean[df_clean["week"].isin(full_weeks)].copy()

print(f"Trimmed partial boundary weeks: {n_before:,} → {len(df_clean):,} tickets")
print(f"Analysis window: {df_clean['week'].min().date()} → {df_clean['week'].max().date()}")
print(f"Complete weeks: {df_clean['week'].nunique()}")

Date range: 2025-01-01 → 2025-12-31
Trimmed partial boundary weeks: 274,550 → 270,559 tickets
Analysis window: 2025-01-06 → 2025-12-22
Complete weeks: 51


## A2 — Field Normalisation

Last mutation of `df_clean`. After this cell, the dataframe is frozen.

In [7]:
# ── Standardise case and strip whitespace ─────────────────────────────────────
for col in ["Category", "Subcategory", "Service"]:
    df_clean[col] = df_clean[col].str.strip().str.title()

# ── Quick missingness check on pipeline-critical fields ───────────────────────
critical = ["Service", "Category", "Subcategory", "Created", "Number"]
missing = df_clean[critical].isna().mean().round(4)
print("Missingness in pipeline-critical fields:")
print(missing.to_string())

# ── State and priority distributions ──────────────────────────────────────────
print(f"\nState distribution:")
print(df_clean["State"].value_counts(normalize=True).round(3).to_string())
print(f"\nPriority distribution:")
print(df_clean["Priority"].value_counts().to_string())

Missingness in pipeline-critical fields:
Service        0.0575
Category       0.0006
Subcategory    0.0006
Created        0.0000
Number         0.0000

State distribution:
State
Closed         0.923
Canceled       0.059
In Progress    0.014
New            0.002
Resolved       0.001
On Hold        0.001

Priority distribution:
Priority
5 - Planning    244430
3 - Moderate     12217
1 - Critical      7705
4 - Low           3258
2 - High          2949


## A3 — Weekly Aggregation (Single Canonical Build)

One build of `service_weekly`. Used by both Chunk A (exploratory) and Chunk B (spike detection).

Outputs:
- `service_counts` — full service-level volume ranking (all services)
- `service_weekly` — per-service weekly aggregation for top-N services
- `weekly_global` — global weekly time series for trend/seasonal context

In [8]:
# ── Service volume ranking ────────────────────────────────────────────────────
service_counts = (
    df_clean["Service"]
    .value_counts()
    .reset_index()
)
service_counts.columns = ["Service", "count"]
service_counts["cum_share"] = service_counts["count"].cumsum() / service_counts["count"].sum()

top_services_list = service_counts.head(CONFIG["top_n_services"])["Service"].tolist()

print(f"Unique services: {len(service_counts):,}")
print(f"Top-{CONFIG['top_n_services']} coverage: {service_counts.head(CONFIG['top_n_services'])['cum_share'].iloc[-1]:.1%}")
print(f"\nTop 10 services:")
print(service_counts.head(10).to_string(index=False))

Unique services: 1,002
Top-60 coverage: 80.8%

Top 10 services:
                           Service  count  cum_share
           Device Support Services  50579   0.198358
             Radiology It Services  14892   0.256760
     Identity Lifecycle Management  14523   0.313715
           Michart Security Access   7635   0.343658
         Redcap (Academic License)   7288   0.372240
                   Paging Services   6521   0.397813
                   Ambulatory Care   5367   0.418861
    Coreimage Engineering Services   4170   0.435215
Academic It Classroom Technologies   4024   0.450996
                 Printing Services   3903   0.466302


In [9]:
# ── Per-service weekly aggregation ────────────────────────────────────────────
# Single canonical build: ticket_count + median_MTTR per (Service, week).
# Consumed by A5 (exploratory charts) and B2 (spike detection).

service_weekly = (
    df_clean[df_clean["Service"].isin(top_services_list)]
    .groupby(["Service", "week"])
    .agg(
        ticket_count=("Number",     "count"),
        median_MTTR =("MTTR_hours", "median"),
    )
    .reset_index()
    .sort_values(["Service", "week"])
)

print(f"service_weekly shape: {service_weekly.shape}")
print(f"Services: {service_weekly['Service'].nunique()}")
print(f"Weeks:    {service_weekly['week'].nunique()}")

service_weekly shape: (3045, 4)
Services: 60
Weeks:    51


In [10]:
# ── Global weekly time series ─────────────────────────────────────────────────
# For trend/seasonal analysis only; not consumed by spike detection.

weekly_global = (
    df_clean
    .groupby("week")
    .agg(
        ticket_count=("Number", "count"),
        median_MTTR =("MTTR_hours", "median"),
    )
    .reset_index()
    .sort_values("week")
)

mean_w = weekly_global["ticket_count"].mean()
cv     = weekly_global["ticket_count"].std() / mean_w
vmi    = weekly_global["ticket_count"].var() / mean_w

print(f"Weekly mean:  {mean_w:.0f} tickets")
print(f"CV:           {cv:.3f}")
print(f"Variance/mean (overdispersion index): {vmi:.1f}")

Weekly mean:  5305 tickets
CV:           0.129
Variance/mean (overdispersion index): 88.4


## A3.5 ─ Threshold Context

In [11]:
# ── Diagnostic: MTTR distribution context for threshold calibration ───────────
# These distributions inform the automated_mttr_ceil and min_mttr_abs thresholds.

resolved = df_clean[df_clean["MTTR_hours"].notna() & (df_clean["MTTR_hours"] >= 0)]

print("── Global MTTR distribution (hours) ──")
print(resolved["MTTR_hours"].describe(percentiles=[.05, .10, .25, .50, .75, .90, .95]).round(2).to_string())

print(f"\n── Fast-resolution breakdown ──")
for cutoff in [0.05, 0.1, 0.25, 0.5, 1.0]:
    n = (resolved["MTTR_hours"] <= cutoff).sum()
    print(f"  MTTR ≤ {cutoff:5.2f}h: {n:>7,} tickets ({n/len(resolved):.1%})")

# Per-service median MTTR for spike-week context
print(f"\n── Weekly median MTTR distribution (service_weekly) ──")
print(service_weekly["median_MTTR"].describe(percentiles=[.05, .10, .25, .50]).round(2).to_string())

low_mttr_weeks = service_weekly[service_weekly["median_MTTR"] < 0.5]
print(f"\n  Service-weeks with median MTTR < 0.5h: {len(low_mttr_weeks)} ({len(low_mttr_weeks)/len(service_weekly):.1%})")
print(f"  Service-weeks with median MTTR < 0.1h: {(service_weekly['median_MTTR'] < 0.1).sum()}")

── Global MTTR distribution (hours) ──
count    250139.00
mean        145.30
std         503.78
min           0.00
5%            0.00
10%           0.00
25%           0.00
50%           1.64
75%          67.84
90%         328.84
95%         715.87
max        9554.08

── Fast-resolution breakdown ──
  MTTR ≤  0.05h:  71,287 tickets (28.5%)
  MTTR ≤  0.10h:  78,803 tickets (31.5%)
  MTTR ≤  0.25h:  91,997 tickets (36.8%)
  MTTR ≤  0.50h: 104,127 tickets (41.6%)
  MTTR ≤  1.00h: 116,392 tickets (46.5%)

── Weekly median MTTR distribution (service_weekly) ──
count    3045.00
mean       48.31
std       186.87
min         0.00
5%          0.00
10%         0.00
25%         0.00
50%         1.75
max      4362.19

  Service-weeks with median MTTR < 0.5h: 1327 (43.6%)
  Service-weeks with median MTTR < 0.1h: 1124


In [12]:
# ── Diagnostic: Volume distribution context ───────────────────────────────────
# These inform vol_ratio_threshold, min_vol_abs, and min_weekly_tickets.

print("── Weekly ticket count distribution (service_weekly) ──")
print(service_weekly["ticket_count"].describe(percentiles=[.05, .10, .25, .50, .75, .90, .95]).round(0).to_string())

print(f"\n── Volume floor context ──")
for floor in [10, 20, 40, 60]:
    n = (service_weekly["ticket_count"] < floor).sum()
    print(f"  Weeks with < {floor:>3} tickets: {n:>5} ({n/len(service_weekly):.1%})")

print(f"\n── Excess context (what does a 1.5x spike look like?) ──")
# For a reader to judge whether min_vol_abs=40 is reasonable
medians = service_weekly.groupby("Service")["ticket_count"].median()
print(f"  Service median volume: min={medians.min():.0f}, p25={medians.quantile(.25):.0f}, "
      f"p50={medians.quantile(.5):.0f}, p75={medians.quantile(.75):.0f}, max={medians.max():.0f}")
print(f"  A 1.5x spike on p25 service ({medians.quantile(.25):.0f}/wk) = excess of {medians.quantile(.25) * 0.5:.0f} tickets")
print(f"  A 1.5x spike on p50 service ({medians.quantile(.5):.0f}/wk) = excess of {medians.quantile(.5) * 0.5:.0f} tickets")

── Weekly ticket count distribution (service_weekly) ──
count    3045.0
mean       68.0
std       141.0
min         1.0
5%         11.0
10%        13.0
25%        20.0
50%        31.0
75%        62.0
90%       119.0
95%       187.0
max      1582.0

── Volume floor context ──
  Weeks with <  10 tickets:   111 (3.6%)
  Weeks with <  20 tickets:   746 (24.5%)
  Weeks with <  40 tickets:  1819 (59.7%)
  Weeks with <  60 tickets:  2237 (73.5%)

── Excess context (what does a 1.5x spike look like?) ──
  Service median volume: min=12, p25=21, p50=28, p75=60, max=1013
  A 1.5x spike on p25 service (21/wk) = excess of 10 tickets
  A 1.5x spike on p50 service (28/wk) = excess of 14 tickets


## A4 — Caller Repetition & Workflow Noise

Two complementary analyses:
1. **Caller concentration** — Gini coefficient + top-5% share per service.
   Surfaces services with structurally unequal submission patterns.
2. **Duplicate detection** — same caller / service / subcategory within 72h.
   Separates human re-submissions from automated ticket streams.

Outputs:
- `caller_concentration_df` — per-service caller inequality metrics
- `duplicate_clusters_df` — cluster membership table (human + automated)
- Flags added to `df_clean`: `is_duplicate_cluster`, `is_automated_caller`

In [13]:
# ── A4.1: Caller Concentration ────────────────────────────────────────────────

def gini(array):
    """Gini coefficient: 0 = perfectly equal, 1 = maximum concentration."""
    array = np.sort(array.astype(float))
    n = len(array)
    if n == 0 or array.sum() == 0:
        return 0.0
    cum = np.cumsum(array)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n


MIN_SVC = CONFIG["min_service_tickets"]

# Filter services with enough data
service_totals = df_clean.groupby("Service").size().reset_index(name="total_tickets")
valid_services = service_totals[service_totals["total_tickets"] >= MIN_SVC]["Service"]
df_valid = df_clean[df_clean["Service"].isin(valid_services)]

# Per (Service, Caller) ticket counts
svc_caller = (
    df_valid.groupby(["Service", "Caller"])
    .size()
    .reset_index(name="ticket_count")
)

# Gini coefficient per service
gini_df = (
    svc_caller.groupby("Service")["ticket_count"]
    .apply(lambda x: gini(x.values))
    .reset_index(name="gini")
)

# Top-5% caller share per service
n_callers_df = svc_caller.groupby("Service")["Caller"].nunique().reset_index(name="n_callers")
svc_caller["rank"] = svc_caller.groupby("Service")["ticket_count"].rank(method="first", ascending=False)
svc_caller = svc_caller.merge(n_callers_df, on="Service")
svc_caller["top_5pct_cutoff"] = np.maximum(1, np.ceil(svc_caller["n_callers"] * 0.05))
svc_caller["is_top_5pct"] = svc_caller["rank"] <= svc_caller["top_5pct_cutoff"]

top5_tickets = (
    svc_caller[svc_caller["is_top_5pct"]]
    .groupby("Service")["ticket_count"]
    .sum()
    .reset_index(name="top_5pct_tickets")
)

# Assemble
caller_concentration_df = (
    service_totals[service_totals["total_tickets"] >= MIN_SVC]
    .merge(n_callers_df, on="Service")
    .merge(top5_tickets, on="Service", how="left")
    .merge(gini_df, on="Service")
)
caller_concentration_df["top_5pct_tickets"] = caller_concentration_df["top_5pct_tickets"].fillna(0)
caller_concentration_df["top_5pct_share"] = (
    caller_concentration_df["top_5pct_tickets"] / caller_concentration_df["total_tickets"]
).round(3)
caller_concentration_df["gini"] = caller_concentration_df["gini"].round(3)

print(f"Services analysed (≥{MIN_SVC} tickets): {len(caller_concentration_df)}")
print(f"\nTop 15 by Gini:")
print(
    caller_concentration_df
    .sort_values("gini", ascending=False)
    .head(15)
    [["Service", "total_tickets", "n_callers", "gini", "top_5pct_share"]]
    .to_string(index=False)
)

Services analysed (≥50 tickets): 255

Top 15 by Gini:
                             Service  total_tickets  n_callers  gini  top_5pct_share
  Academic It Classroom Technologies           4024        272 0.897           0.917
                     Pathology Atlas            752         12 0.861           0.878
  Cloud Kubernetes Container Hosting            364         32 0.852           0.835
                     Mobileview Rfid            340         27 0.830           0.791
                        Service Desk           3766        642 0.828           0.838
                      Patient Portal            116          9 0.814           0.914
       Identity Lifecycle Management          14523        969 0.807           0.593
               Patient Entertainment            310         58 0.798           0.816
                  Hfac Ipaas - Boomi            462          5 0.782           0.970
               Radiology It Services          14892       1080 0.782           0.599
           

In [14]:
# ── A4.2: Duplicate Detection ─────────────────────────────────────────────────

GAP_H    = CONFIG["duplicate_gap_hours"]
AUTO_RE  = CONFIG["automated_caller_re"]
GENERIC  = CONFIG["generic_callers"]
GRP_COLS = ["Caller", "Service", "Subcategory"]

# Filter
dup_df = (
    df_clean.dropna(subset=["Created", "Caller", "Service", "Subcategory"])
    .copy()
)
dup_df = dup_df[~dup_df["Caller"].isin(GENERIC)]

# Split human / automated
is_auto = dup_df["Caller"].str.lower().str.contains(AUTO_RE, regex=True, na=False)
dup_human     = dup_df[~is_auto].copy()
dup_automated = dup_df[is_auto].copy()

print(f"Tickets after generic exclusions: {len(dup_df):,}")
print(f"  Human callers:     {len(dup_human):,}")
print(f"  Automated callers: {len(dup_automated):,}")


def build_clusters(df_in):
    """Assign cluster IDs within each (Caller, Service, Subcategory) group."""
    out = df_in.sort_values(GRP_COLS + ["Created"]).reset_index(drop=True)
    out["prev_created"] = out.groupby(GRP_COLS)["Created"].shift(1)
    out["gap_hours"] = (out["Created"] - out["prev_created"]).dt.total_seconds() / 3600
    out["new_cluster"] = out["gap_hours"].isna() | (out["gap_hours"] > GAP_H)
    out["cluster_id"]  = out.groupby(GRP_COLS)["new_cluster"].cumsum()
    return out


def aggregate_clusters(df_in):
    """Aggregate per-cluster stats; keep only multi-ticket clusters."""
    clusters = (
        df_in.groupby(GRP_COLS + ["cluster_id"])
        .agg(
            ticket_count  =("Number",  "count"),
            first_created =("Created", "min"),
            last_created  =("Created", "max"),
            ticket_numbers=("Number",  list),
        )
        .reset_index()
    )
    clusters["duration_hours"] = (
        (clusters["last_created"] - clusters["first_created"]).dt.total_seconds() / 3600
    )
    return clusters[clusters["ticket_count"] > 1].copy()


dup_human     = build_clusters(dup_human)
dup_automated = build_clusters(dup_automated)
human_clusters     = aggregate_clusters(dup_human)
automated_clusters = aggregate_clusters(dup_automated)


# Combine into single output table
human_clusters["caller_type"]     = "human"
automated_clusters["caller_type"] = "automated"
automated_clusters["pattern"]     = "Automated stream"

duplicate_clusters_df = pd.concat(
    [human_clusters, automated_clusters], ignore_index=True
).sort_values(["ticket_count", "duration_hours"], ascending=[False, True])

print(f"\nHuman duplicate clusters:    {len(human_clusters):,}")
print(f"Automated ticket streams:    {len(automated_clusters):,}")

/var/folders/sd/5vy204s55_ggyjh65z1tlxg00000gn/T/ipykernel_25659/194716591.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_auto = dup_df["Caller"].str.lower().str.contains(AUTO_RE, regex=True, na=False)


Tickets after generic exclusions: 242,004
  Human callers:     237,181
  Automated callers: 4,823

Human duplicate clusters:    16,285
Automated ticket streams:    465


In [15]:
# ── A4.3: Flag df_clean ───────────────────────────────────────────────────────
# These are the LAST mutations to df_clean.

# Collect all ticket numbers that belong to a duplicate cluster
dup_ticket_numbers = set()
for numbers in duplicate_clusters_df["ticket_numbers"]:
    dup_ticket_numbers.update(numbers)

df_clean["is_duplicate_cluster"] = df_clean["Number"].isin(dup_ticket_numbers)

# Automated caller flag (matches the same regex used in duplicate detection)
df_clean["is_automated_caller"] = (
    df_clean["Caller"]
    .fillna("")
    .str.lower()
    .str.contains(CONFIG["automated_caller_re"], regex=True)
)

n_dup  = df_clean["is_duplicate_cluster"].sum()
n_auto = df_clean["is_automated_caller"].sum()
print(f"Tickets flagged as duplicate cluster members: {n_dup:,} ({n_dup/len(df_clean):.1%})")
print(f"Tickets from automated callers:               {n_auto:,} ({n_auto/len(df_clean):.1%})")
print(f"\ndf_clean is now frozen. Shape: {df_clean.shape}")

Tickets flagged as duplicate cluster members: 66,874 (24.7%)
Tickets from automated callers:               5,052 (1.9%)

df_clean is now frozen. Shape: (270559, 34)


/var/folders/sd/5vy204s55_ggyjh65z1tlxg00000gn/T/ipykernel_25659/2814239250.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(CONFIG["automated_caller_re"], regex=True)


## A5 — Exploratory Context (Non-Pipeline)

These cells are analytical context only. They read from `df_clean`, `service_weekly`,
and `weekly_global` but produce no outputs consumed downstream.
Collapse or skip without breaking the pipeline.

### A5.1 — Global Weekly Trend

In [16]:
# ── Global weekly trend with rolling average ─────────────────────────────────
weekly_global["rolling_3w"] = weekly_global["ticket_count"].rolling(3, center=True).mean()

base = alt.Chart(weekly_global).encode(
    x=alt.X("week:T", title="Week", axis=alt.Axis(labelAngle=0, tickCount=12))
)

line_raw = base.mark_line(color="#BBBBBB", strokeWidth=1, opacity=0.6).encode(
    y=alt.Y("ticket_count:Q", title="Incidents created")
)
line_trend = base.mark_line(color="#E45756", strokeWidth=2.5).encode(y="rolling_3w:Q")

chart1 = (line_raw + line_trend).properties(
    width=620, height=300,
    title="Weekly Ticket Volume (raw + 3-week rolling average)"
)

chart1.save("weekly_trend.html")

chart1

alt.LayerChart(...)

### A5.2 — Intraday & Day-of-Week Patterns

In [17]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

hour_dow = (
    df_clean.groupby(["dow", "hour"])
    .size()
    .reset_index(name="count")
)
hour_dow["pct_within_day"] = (
    hour_dow["count"] / hour_dow.groupby("dow")["count"].transform("sum")
)

chart2 = alt.Chart(hour_dow).mark_rect().encode(
    x=alt.X("hour:O", title="Hour of day"),
    y=alt.Y("dow:O", sort=dow_order, title="Day of week"),
    color=alt.Color("pct_within_day:Q", title="Proportion of daily incidents",
                    scale=alt.Scale(scheme="blues")),
    tooltip=["dow:N", "hour:O", alt.Tooltip("pct_within_day:Q", format=".3f")]
).properties(title="Incident Distribution Within Each Day (Normalised)", width=500, height=200)

chart2.save("hour_dow_heatmap.html")

chart2

alt.Chart(...)

### A5.3 — Seasonal Decomposition

In [18]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Global decomposition
weekly_ts = weekly_global.set_index("week")["ticket_count"]

decomp = seasonal_decompose(weekly_ts, model="additive", period=13, extrapolate_trend="freq")

decomp_df = pd.DataFrame({
    "week":     weekly_ts.index,
    "observed": weekly_ts.values,
    "trend":    decomp.trend,
    "seasonal": decomp.seasonal,
    "resid":    decomp.resid,
})

seasonal_strength = decomp_df["seasonal"].std() / decomp_df["observed"].std()
resid_strength    = decomp_df["resid"].std()    / decomp_df["observed"].std()

print(f"Seasonal strength (std ratio): {seasonal_strength:.3f}")
print(f"Residual strength (std ratio): {resid_strength:.3f}")

# Chart: observed + seasonal overlay
base = alt.Chart(decomp_df).encode(
    x=alt.X("week:T", title="Week", axis=alt.Axis(labelAngle=-30, tickCount=12))
)
obs_line = base.mark_line(color="#4c78a8", strokeWidth=1.5, opacity=0.7).encode(
    y=alt.Y("observed:Q", title="Tickets per week")
)
sea_line = base.mark_line(color="#E45756", strokeWidth=2).encode(
    y=alt.Y("seasonal:Q", title="Seasonal component")
)
obs_line.properties(width=620, height=200, title="Observed Volume") | \
sea_line.properties(width=620, height=200, title="Seasonal Component")

Seasonal strength (std ratio): 0.551
Residual strength (std ratio): 0.810


alt.HConcatChart(...)

### A5.4 — Workload Concentration

In [19]:
# ── Pareto curve ──────────────────────────────────────────────────────────────
pareto = service_counts.copy()
pareto["rank"] = range(1, len(pareto) + 1)

alt.Chart(pareto).mark_line(point=False).encode(
    x=alt.X("rank:Q", title="Service rank", scale=alt.Scale(type="log")),
    y=alt.Y("cum_share:Q", title="Cumulative share of tickets",
            axis=alt.Axis(format=".0%")),
    tooltip=["Service:N", alt.Tooltip("cum_share:Q", format=".1%")]
).properties(width=500, height=300, title="Service Volume — Pareto Curve")

alt.Chart(...)

In [20]:
# ── Service × Subcategory volume heatmap (top 10 services) ────────────────────
top10_services = service_counts.head(10)["Service"].tolist()

svc_subcat = (
    df_clean[df_clean["Service"].isin(top10_services) & df_clean["Subcategory"].notnull()]
    .groupby(["Service", "Subcategory"])
    .size()
    .reset_index(name="count")
)

chart3 = alt.Chart(svc_subcat).mark_rect().encode(
    x=alt.X("Subcategory:N", title="Subcategory"),
    y=alt.Y("Service:N", sort=top10_services, title="Service"),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="blues"), title="Tickets"),
    tooltip=["Service:N", "Subcategory:N", "count:Q"]
).properties(width=500, height=300, title="Ticket Volume: Service × Subcategory (Top 10)")

chart3.save("service_subcat_heatmap.html")

chart3

alt.Chart(...)

In [21]:
# ── Service × Subcategory MTTR heatmap (top 10 services) ─────────────────────
svc_subcat_mttr = (
    df_clean[
        df_clean["Service"].isin(top10_services)
        & df_clean["Subcategory"].notnull()
        & df_clean["MTTR_hours"].notnull()
    ]
    .groupby(["Service", "Subcategory"])
    .agg(
        median_MTTR=("MTTR_hours", "median"),
        count=("Number", "count")
    )
    .reset_index()
)

# Filter sparse cells — fewer than 30 tickets gives unstable medians
svc_subcat_mttr = svc_subcat_mttr[svc_subcat_mttr["count"] >= 30].copy()

# Log transform so extreme MTTR cells do not wash out the mid-range
svc_subcat_mttr["mttr_log"] = np.log1p(svc_subcat_mttr["median_MTTR"])

chart4 = alt.Chart(svc_subcat_mttr).mark_rect().encode(
    x=alt.X("Subcategory:N", title="Subcategory"),
    y=alt.Y("Service:N", sort=top10_services, title="Service"),
    color=alt.Color(
        "mttr_log:Q",
        scale=alt.Scale(scheme="oranges"),
        title="Median MTTR (log h)"
    ),
    tooltip=[
        "Service:N",
        "Subcategory:N",
        alt.Tooltip("median_MTTR:Q", title="Median MTTR (h)", format=".1f"),
        alt.Tooltip("count:Q", title="Tickets")
    ]
).properties(
    width=500,
    height=300,
    title="Median MTTR: Service × Subcategory (Top 10, log scale)"
)

chart4.save("service_subcat_mttr_heatmap.html")

chart4

alt.Chart(...)

### A5.5 — Efficiency: Volume × MTTR × Priority

In [22]:
# ── Impact score: Volume × Median MTTR per (Service, Subcategory) ─────────────
impact_df = (
    df_clean.dropna(subset=["MTTR_hours", "Service", "Subcategory"])
    .groupby(["Service", "Subcategory"])
    .agg(
        incidents  =("MTTR_hours", "count"),
        median_MTTR=("MTTR_hours", "median"),
    )
    .reset_index()
)
impact_df["impact"] = impact_df["incidents"] * impact_df["median_MTTR"]
impact_df = impact_df.sort_values("impact", ascending=False).reset_index(drop=True)

print(f"Service×Subcategory pairs: {len(impact_df):,}")
print(f"\nTop 20 by impact score:")
table = impact_df.head(20).copy()
table["incidents"]   = (table["incidents"] / 1000).round(1).astype(str) + "k"
table["median_MTTR"] = table["median_MTTR"].round(1).astype(str) + "h"
table["impact"]      = table["impact"].round(0).astype(int)
print(table[["Service", "Subcategory", "incidents", "median_MTTR", "impact"]].to_string(index=False))

Service×Subcategory pairs: 3,552

Top 20 by impact score:
                           Service     Subcategory incidents median_MTTR  impact
    Coreimage Engineering Services           Other      1.4k      621.3h  885360
            Release Of Information           Other      2.2k      142.6h  311974
                   Ambulatory Care Move/Change/Add      1.7k      161.7h  278018
           Device Support Services           Other      7.5k       23.9h  178755
      Continuing Medical Education  Inquiry / Help      1.9k       77.7h  144674
           Device Support Services Move/Change/Add      7.7k       16.3h  125556
      Pharmacy Management Services Move/Change/Add      0.5k      241.6h  120780
           Device Support Services System Degraded     15.2k        7.6h  115929
                         Reporting  Inquiry / Help      0.3k      291.3h   97884
                 Database Services Move/Change/Add      0.8k      112.8h   95887
          Research System Services Move/Change/Add 

In [23]:
# ── Volume × MTTR × Priority by (Service, Subcategory) ──────────────────────
impact_df = (
    df_clean.dropna(subset=["MTTR_hours", "Service", "Subcategory"])
    .groupby(["Service", "Subcategory"])
    .agg(
        incidents=("MTTR_hours", "count"),
        median_MTTR=("MTTR_hours", "median"),
        high_priority_ratio=("Priority", lambda x: x.isin(["1 - Critical", "2 - High"]).mean())
    )
    .reset_index()
)

# Optional: filter sparse cells so tiny groups do not dominate interpretation
impact_df = impact_df[impact_df["incidents"] >= 30].copy()

# Impact score
impact_df["impact"] = impact_df["incidents"] * impact_df["median_MTTR"]

# Display cap for plotting only
mttr_cap = impact_df["median_MTTR"].quantile(0.95)
impact_df["median_MTTR_display"] = impact_df["median_MTTR"].clip(upper=mttr_cap)

# Thresholds for quadrant guides
volume_threshold = impact_df["incidents"].median()
mttr_threshold   = impact_df["median_MTTR_display"].median()

# Priority color scale range
hp_min = impact_df["high_priority_ratio"].min()
hp_max = impact_df["high_priority_ratio"].max()

# Labels: top 10 pairs by impact
labels_df = impact_df.nlargest(10, "impact").copy()
labels_df["pair_label"] = labels_df["Service"] + " × " + labels_df["Subcategory"]

# Quadrant lines first so points render on top
quadrant_h = alt.Chart(pd.DataFrame({"y": [mttr_threshold]})).mark_rule(
    strokeDash=[6, 3], color="#888888", strokeWidth=1.5
).encode(
    y="y:Q"
)

quadrant_v = alt.Chart(pd.DataFrame({"x": [volume_threshold]})).mark_rule(
    strokeDash=[6, 3], color="#888888", strokeWidth=1.5
).encode(
    x="x:Q"
)

points = alt.Chart(impact_df).mark_circle(size=70, opacity=0.75).encode(
    x=alt.X(
        "incidents:Q",
        scale=alt.Scale(type="log"),
        title="Ticket volume (log scale)"
    ),
    y=alt.Y(
        "median_MTTR_display:Q",
        scale=alt.Scale(type="sqrt", clamp=True),
        title=f"Median MTTR (hours, sqrt scale, capped at {mttr_cap:.0f}h)"
    ),
    color=alt.Color(
        "high_priority_ratio:Q",
        title="High-priority share",
        scale=alt.Scale(scheme="oranges", domain=[hp_min, hp_max]),
        legend=alt.Legend(format=".1%")
    ),
    tooltip=[
        "Service:N",
        "Subcategory:N",
        alt.Tooltip("incidents:Q", title="Tickets", format=","),
        alt.Tooltip("median_MTTR:Q", title="Median MTTR (hrs, uncapped)", format=".1f"),
        alt.Tooltip("high_priority_ratio:Q", title="High-priority share", format=".1%"),
        alt.Tooltip("impact:Q", title="Impact score", format=",.0f")
    ]
)

labels = alt.Chart(labels_df).mark_text(
    align="left", dx=6, dy=-5, fontSize=10
).encode(
    x="incidents:Q",
    y="median_MTTR_display:Q",
    text="pair_label:N"
)

# Layer order: quadrant lines → points → labels
chart5 = (quadrant_h + quadrant_v + points + labels).properties(
    title="Volume × MTTR × Priority by Service × Subcategory",
    width=640,
    height=400
)

chart5.save("volume_mttr_priority_scatter.html")
chart5

alt.LayerChart(...)

## Chunk A — Output Summary

| Object | Shape | Description |
|---|---|---|
| `df_clean` | (tickets × cols) | One row per incident, all temporal/categorical fields clean, `is_duplicate_cluster` and `is_automated_caller` flags |
| `service_weekly` | (service-weeks × 4) | `Service`, `week`, `ticket_count`, `median_MTTR` for top-N services |
| `service_counts` | (services × 3) | `Service`, `count`, `cum_share` — full volume ranking |
| `weekly_global` | (weeks × 3) | Global weekly aggregation |
| `caller_concentration_df` | (services × 6) | Per-service Gini + top-5% share |
| `duplicate_clusters_df` | (clusters × cols) | Duplicate cluster membership (human + automated) |
| `CONFIG` | dict | All pipeline thresholds — consumed by Chunks B and C |

Chunk B reads `df_clean`, `service_weekly`, `service_counts`, and `CONFIG`.
No other objects are required.

In [24]:
# ── Verify outputs exist ──────────────────────────────────────────────────────
print("Chunk A outputs:")
for name, obj in [
    ("df_clean",                df_clean),
    ("service_weekly",          service_weekly),
    ("service_counts",          service_counts),
    ("weekly_global",           weekly_global),
    ("caller_concentration_df", caller_concentration_df),
    ("duplicate_clusters_df",   duplicate_clusters_df),
]:
    if isinstance(obj, pd.DataFrame):
        print(f"  {name:30s} {str(obj.shape):>15s}")
    else:
        print(f"  {name:30s} {type(obj).__name__}")
print(f"  {'CONFIG':30s} {len(CONFIG)} keys")
print("\nChunk A complete. Ready for Chunk B.")

Chunk A outputs:
  df_clean                          (270559, 34)
  service_weekly                       (3045, 4)
  service_counts                       (1002, 3)
  weekly_global                          (51, 4)
  caller_concentration_df               (255, 6)
  duplicate_clusters_df              (16750, 11)
  CONFIG                         23 keys

Chunk A complete. Ready for Chunk B.


---

# Stage 2 - Ranking Candidates

Two independent arms produce candidates:
- **B1 Persistent burden (static):** Service × Subcategory pairs with high structural impact.
  Measures *magnitude* — how much total resolution effort a category consumes.
- **B2–B3 Episodic disruptions (temporal):** Service-weeks where demand or resolution time
  deviates significantly from the rolling baseline.
  Measures *deviation* — how far a week is from what's normal for that service.

These are deliberately kept distinct: a service can have high persistent burden
without ever spiking (steady large load), and a service can spike without
ranking high on persistent burden (small service, sudden anomaly).

B4 assembles both arms into a unified `candidate_list`.
B5 screens each candidate to determine whether structured fields already
explain the pattern — NLP is only triggered where they don't.

## B1 — Persistent Burden (Static Arm)

Ranks Service × Subcategory pairs by **impact = volume × median MTTR**.
This captures structural load — pairs that consume the most total resolution effort
regardless of temporal pattern.

Distinction from B2–B3: persistent burden does not require a spike.
A pair can rank #1 here while showing perfectly flat weekly volume.

In [25]:
# ── Static impact scoring ─────────────────────────────────────────────────────
static_df = (
    df_clean.dropna(subset=["MTTR_hours", "Service", "Subcategory"])
    .groupby(["Service", "Subcategory"])
    .agg(
        volume     =("Number",     "count"),
        median_MTTR=("MTTR_hours", "median"),
    )
    .reset_index()
)

static_df["impact_score"] = static_df["volume"] * static_df["median_MTTR"]

# ── Rank and threshold ────────────────────────────────────────────────────────
# Single filter: top-N by impact, with a volume floor.
# The volume floor (≥ 200 tickets) excludes tiny-sample pairs where a single
# extreme MTTR inflates the impact score beyond what the data supports.
# No percentile floor — p90 was too permissive (admitted ~355 of 3,552 pairs).

STATIC_TOP_N     = 20
MIN_STATIC_VOL   = 200    # minimum tickets to qualify

static_df["rank"] = static_df["impact_score"].rank(ascending=False, method="min").astype(int)
static_df = static_df.sort_values("impact_score", ascending=False).reset_index(drop=True)

static_candidates = static_df[
    (static_df["rank"] <= STATIC_TOP_N)
    & (static_df["volume"] >= MIN_STATIC_VOL)
].copy()

print(f"Total Service×Subcategory pairs: {len(static_df):,}")
print(f"Volume floor:                    ≥ {MIN_STATIC_VOL}")
print(f"Static candidates:               {len(static_candidates)}")
print(f"\nTop 20 by impact (vol ≥ {MIN_STATIC_VOL}):")
display_cols = ["Service", "Subcategory", "volume", "median_MTTR", "impact_score", "rank"]
top20 = static_candidates.head(20).copy()
top20["median_MTTR"] = top20["median_MTTR"].round(1)
top20["impact_score"] = top20["impact_score"].round(0).astype(int)
print(top20[display_cols].to_string(index=False))

Total Service×Subcategory pairs: 3,552
Volume floor:                    ≥ 200
Static candidates:               17

Top 20 by impact (vol ≥ 200):
                           Service     Subcategory  volume  median_MTTR  impact_score  rank
    Coreimage Engineering Services           Other    1425        621.3        885360     1
            Release Of Information           Other    2188        142.6        311974     2
                   Ambulatory Care Move/Change/Add    1719        161.7        278018     3
           Device Support Services           Other    7478         23.9        178755     4
      Continuing Medical Education  Inquiry / Help    1861         77.7        144674     5
           Device Support Services Move/Change/Add    7701         16.3        125556     6
      Pharmacy Management Services Move/Change/Add     500        241.6        120780     7
           Device Support Services System Degraded   15214          7.6        115929     8
                         Re

## B2 — Baseline & Spike Detection (Temporal Arm)

Augments `service_weekly` (built in A3) with rolling baselines and spike flags.

**Volume baseline:** Rolling median over the prior 8 weeks (shift-1).
Median is robust to outliers — a single extreme week cannot distort it.

**MTTR baseline:** Rolling 90th percentile of weekly median MTTR.
p90 is more robust than mean ± SD for heavy-tailed MTTR distributions.

**Spike criteria** (both must hold for each type):
- **Volume spike:** ratio ≥ 1.5 AND excess ≥ 15% of baseline AND excess ≥ 40 tickets
- **MTTR spike:** ratio ≥ 1.5 AND excess ≥ 10 hours above p90
- Both types require ≥ 20 tickets in the week (excludes near-empty weeks)

In [26]:
# ── Rolling baselines ─────────────────────────────────────────────────────────
# Volume: rolling median (robust to spike contamination)
service_weekly["roll_median_vol"] = (
    service_weekly.groupby("Service")["ticket_count"]
    .transform(
        lambda x: x.shift(1)
        .rolling(CONFIG["baseline_weeks"], min_periods=CONFIG["min_baseline_periods"])
        .median()
    )
)

# MTTR: rolling 90th percentile
service_weekly["mttr_p90"] = (
    service_weekly.groupby("Service")["median_MTTR"]
    .transform(
        lambda x: x.shift(1)
        .rolling(CONFIG["baseline_weeks"], min_periods=CONFIG["min_baseline_periods"])
        .quantile(0.9)
    )
)

# ── Derived comparison columns ────────────────────────────────────────────────
service_weekly["vol_excess"]  = service_weekly["ticket_count"] - service_weekly["roll_median_vol"]
service_weekly["vol_ratio"]   = (
    service_weekly["ticket_count"] / service_weekly["roll_median_vol"].replace(0, np.nan)
)
service_weekly["mttr_excess"] = service_weekly["median_MTTR"] - service_weekly["mttr_p90"]
service_weekly["mttr_ratio"]  = (
    service_weekly["median_MTTR"] / service_weekly["mttr_p90"].replace(0, np.nan)
)

print(f"Baseline columns added to service_weekly. Shape: {service_weekly.shape}")

Baseline columns added to service_weekly. Shape: (3045, 10)


In [27]:
# ── Spike flags ───────────────────────────────────────────────────────────────
C = CONFIG  # shorthand

service_weekly["volume_spike"] = (
    service_weekly["roll_median_vol"].notna()
    & (service_weekly["ticket_count"] >= C["min_weekly_tickets"])
    & (service_weekly["vol_ratio"]    >= C["vol_ratio_threshold"])
    & (service_weekly["vol_excess"]   >= service_weekly["roll_median_vol"] * C["min_vol_rel"])
    & (service_weekly["vol_excess"]   >= C["min_vol_abs"])
)

service_weekly["mttr_spike"] = (
    service_weekly["mttr_p90"].notna()
    & (service_weekly["ticket_count"] >= C["min_weekly_tickets"])
    & (service_weekly["median_MTTR"]  >  service_weekly["mttr_p90"])
    & (service_weekly["mttr_excess"]  >= C["min_mttr_abs"])
    & (service_weekly["mttr_ratio"]   >= C["mttr_ratio_threshold"])
)

service_weekly["any_spike"] = service_weekly["volume_spike"] | service_weekly["mttr_spike"]

# ── Spike type label ──────────────────────────────────────────────────────────
service_weekly["spike_type"] = np.select(
    [
        service_weekly["volume_spike"] & service_weekly["mttr_spike"],
        service_weekly["volume_spike"],
        service_weekly["mttr_spike"],
    ],
    ["Volume + MTTR", "Volume only", "MTTR only"],
    default="Normal",
)

# ── Automated-processing flag ─────────────────────────────────────────────────
# Volume spike weeks with near-zero MTTR are automated batch jobs.
service_weekly["is_automated"] = (
    service_weekly["volume_spike"] & (service_weekly["median_MTTR"] < C["automated_mttr_ceil"])
)

# ── Summary ───────────────────────────────────────────────────────────────────
n_vol  = int(service_weekly["volume_spike"].sum())
n_mttr = int(service_weekly["mttr_spike"].sum())
n_any  = int(service_weekly["any_spike"].sum())
n_auto = int(service_weekly["is_automated"].sum())

print(f"Service-weeks analysed:      {len(service_weekly):,}")
print(f"Volume spike weeks:          {n_vol}  ({n_auto} automated/batch)")
print(f"MTTR spike weeks:            {n_mttr}")
print(f"Either type:                 {n_any}")
print(f"Actionable (excl automated): {n_any - n_auto}")

Service-weeks analysed:      3,045
Volume spike weeks:          64  (29 automated/batch)
MTTR spike weeks:            69
Either type:                 131
Actionable (excl automated): 102


## B3 — Episode Merging & Severity

Consecutive spike weeks for the same service represent a single sustained event.
Merging prevents double-counting and gives a duration signal.

**Severity score** combines three orthogonal signals:
- `log2(peak_ratio)` — how extreme the deviation was (intensity)
- `log1p(total_tickets)` — how many tickets were affected (magnitude)
- `sqrt(spike_weeks)` — how long it lasted (persistence)

The log/sqrt transforms prevent any single signal from dominating.

In [28]:
# ── Episode assignment ─────────────────────────────────────────────────────────

def assign_episodes(spike_array, max_gap=1):
    """
    Merge consecutive spike weeks into episodes.

    Parameters
    ----------
    spike_array : array-like of bool
    max_gap     : int — non-spike weeks allowed within a single episode

    Returns
    -------
    episodes   : array of episode IDs (0 = not part of any episode)
    is_bridged : bool array marking gap weeks inside an episode
    """
    n          = len(spike_array)
    episodes   = np.zeros(n, dtype=int)
    is_bridged = np.zeros(n, dtype=bool)
    episode_id = 0
    in_episode = False
    gap_count  = 0

    for i, is_spike in enumerate(spike_array):
        if is_spike:
            if not in_episode:
                episode_id += 1
                in_episode  = True
            gap_count   = 0
            episodes[i] = episode_id
        else:
            if in_episode:
                gap_count += 1
                if gap_count > max_gap:
                    in_episode = False
                    gap_count  = 0
                else:
                    episodes[i]   = episode_id
                    is_bridged[i] = True
    return episodes, is_bridged


MAX_GAP = CONFIG["episode_max_gap"]

for spike_col, ep_col, br_col in [
    ("volume_spike", "volume_episode", "volume_bridged"),
    ("mttr_spike",   "mttr_episode",   "mttr_bridged"),
]:
    service_weekly[ep_col] = (
        service_weekly.groupby("Service")[spike_col]
        .transform(lambda s: pd.Series(assign_episodes(s.values, MAX_GAP)[0], index=s.index))
    )
    service_weekly[br_col] = (
        service_weekly.groupby("Service")[spike_col]
        .transform(lambda s: pd.Series(assign_episodes(s.values, MAX_GAP)[1], index=s.index))
    )

print("Episode IDs assigned.")
print(f"  Volume episodes: {service_weekly[service_weekly['volume_episode'] > 0]['volume_episode'].nunique()}")
print(f"  MTTR episodes:   {service_weekly[service_weekly['mttr_episode'] > 0]['mttr_episode'].nunique()}")

Episode IDs assigned.
  Volume episodes: 4
  MTTR episodes:   4


In [29]:
# ── Episode summary tables ────────────────────────────────────────────────────

def build_episode_summary(sw, episode_col, bridged_col, metric_col, baseline_col, df_clean):
    """
    Aggregate per-episode statistics from service_weekly.
    Enriches with duplicate share from df_clean for spike deflation.
    """
    mask = sw[episode_col] > 0
    grp  = sw[mask].groupby(["Service", episode_col])

    summary = grp.agg(
        start_week   =("week",         "min"),
        end_week     =("week",         "max"),
        total_rows   =(episode_col,    "count"),
        spike_weeks  =(bridged_col,    lambda x: (~x).sum()),
        peak_value   =(metric_col,     "max"),
        baseline     =(baseline_col,   "median"),
        total_tickets=("ticket_count", "sum"),
        is_automated =("is_automated", "max"),
    ).reset_index()

    summary["peak_ratio"] = (
        summary["peak_value"] / summary["baseline"].replace(0, np.nan)
    ).round(2)

    # ── Duplicate share per episode ───────────────────────────────────────
    # How much of this episode's volume is duplicate cluster tickets?
    # High dup_share means the spike may be caller behavior, not a system event.
    dup_shares = []
    for _, row in summary.iterrows():
        ep_mask = (
            (df_clean["Service"] == row["Service"])
            & (df_clean["week"] >= row["start_week"])
            & (df_clean["week"] <= row["end_week"])
        )
        ep_tickets = df_clean[ep_mask]
        if len(ep_tickets) > 0:
            dup_shares.append(ep_tickets["is_duplicate_cluster"].mean())
        else:
            dup_shares.append(0.0)
    summary["dup_share"] = [round(d, 3) for d in dup_shares]

    # Severity score: intensity × magnitude × persistence
    summary["severity"] = (
        np.log2(summary["peak_ratio"].clip(lower=1.01))
        * np.log1p(summary["total_tickets"])
        * np.sqrt(summary["spike_weeks"])
    )

    return summary.sort_values("severity", ascending=False).reset_index(drop=True)


vol_episodes = build_episode_summary(
    service_weekly, "volume_episode", "volume_bridged",
    "ticket_count", "roll_median_vol", df_clean,
)
mttr_episodes = build_episode_summary(
    service_weekly, "mttr_episode", "mttr_bridged",
    "median_MTTR", "mttr_p90", df_clean,
)

n_vol_ep  = len(vol_episodes)
n_auto_ep = int(vol_episodes["is_automated"].sum())
n_mttr_ep = len(mttr_episodes)

print(f"Volume episodes:  {n_vol_ep}  ({n_auto_ep} automated/batch)")
print(f"MTTR episodes:    {n_mttr_ep}")
print(f"Actionable total: {n_vol_ep - n_auto_ep + n_mttr_ep}")

print(f"\n── Top 15 Volume Episodes (excl automated) ──")
top_vol = vol_episodes[~vol_episodes["is_automated"]].head(15)
print(top_vol[["Service", "start_week", "end_week", "spike_weeks",
               "peak_ratio", "total_tickets", "dup_share", "severity"]].to_string(index=False))

# Flag high-duplicate episodes
high_dup = top_vol[top_vol["dup_share"] >= CONFIG["dup_deflation_warn"]]
if len(high_dup) > 0:
    print(f"\n⚠ {len(high_dup)} episode(s) have ≥{CONFIG['dup_deflation_warn']:.0%} duplicate tickets:")
    for _, r in high_dup.iterrows():
        print(f"  {r['Service']} ({r['start_week'].strftime('%Y-%m-%d')}): {r['dup_share']:.0%} duplicates")

print(f"\n── Top 15 MTTR Episodes ──")
top_mttr = mttr_episodes.head(15)
print(top_mttr[["Service", "start_week", "end_week", "spike_weeks",
                "peak_ratio", "total_tickets", "dup_share", "severity"]].to_string(index=False))

Volume episodes:  38  (19 automated/batch)
MTTR episodes:    64
Actionable total: 83

── Top 15 Volume Episodes (excl automated) ──
                             Service start_week   end_week  spike_weeks  peak_ratio  total_tickets  dup_share   severity
Michart Pharmacy (Willow Ambulatory) 2025-05-12 2025-06-09            4      253.33            595      0.642 102.050535
  Cornerstone Performance Management 2025-05-05 2025-06-30            5        4.47            982      0.105  33.285235
      Coreimage Engineering Services 2025-04-21 2025-04-28            1       22.77           1349      0.047  32.500690
       Identity Lifecycle Management 2025-07-28 2025-08-11            2        6.33           2490      0.917  29.443385
           Outlook For The Web (Owa) 2025-03-03 2025-03-10            1       17.38            359      0.039  24.246959
            Cybersecurity Operations 2025-08-18 2025-09-15            4        3.39            631      0.111  22.716668
                   Da

## B4 — Candidate Assembly

Merges the two arms into a single `candidate_list`:
- **Episodic candidates** from top-K volume + MTTR episodes
- **Static candidates** from B1 persistent burden

Each candidate carries a `candidate_type`:
- `"episodic"` — temporal deviation only
- `"persistent"` — structural burden only
- `"both"` — the service appears in both arms

The distinction matters for Chunk C: episodic candidates scope NLP to a
specific week window; persistent candidates scope NLP to the full population
of that service × subcategory pair.

In [30]:
# ── Temporal candidates ────────────────────────────────────────────────────────
TOP_K = CONFIG["top_k_episodes"]

temporal_vol = (
    vol_episodes[~vol_episodes["is_automated"]]
    .head(TOP_K)
    .assign(spike_arm="volume")
)
temporal_mttr = (
    mttr_episodes
    .head(TOP_K)
    .assign(spike_arm="mttr")
)

temporal_candidates = pd.concat([temporal_vol, temporal_mttr], ignore_index=True)

# Deduplicate: if the same (Service, start_week) appears in both vol and mttr,
# keep both arms noted but merge into one row.
temporal_candidates["candidate_id"] = (
    "ep_" + temporal_candidates["Service"].str.replace(" ", "_")
    + "_" + temporal_candidates["start_week"].dt.strftime("%Y%m%d")
)

def _collapse_arms(grp):
    row = grp.iloc[0].copy()
    arms = set(grp["spike_arm"])
    if arms == {"volume", "mttr"}:
        row["spike_arm"] = "volume+mttr"
    row["severity"] = grp["severity"].max()
    row["dup_share"] = grp["dup_share"].max()
    return row

temporal_candidates = (
    temporal_candidates.groupby("candidate_id", as_index=False)
    .apply(_collapse_arms)
    .reset_index(drop=True)
)

temporal_candidates["candidate_type"] = "episodic"

print(f"Temporal candidates: {len(temporal_candidates)}")
print(temporal_candidates[["candidate_id", "Service", "start_week", "end_week",
                           "spike_arm", "dup_share", "severity"]].to_string(index=False))

Temporal candidates: 29
                                    candidate_id                              Service start_week   end_week   spike_arm  dup_share   severity
      ep_Coreimage_Engineering_Services_20250421       Coreimage Engineering Services 2025-04-21 2025-04-28 volume+mttr      0.047  32.500690
  ep_Cornerstone_Performance_Management_20250505   Cornerstone Performance Management 2025-05-05 2025-06-30      volume      0.105  33.285235
  ep_Cornerstone_Performance_Management_20250728   Cornerstone Performance Management 2025-07-28 2025-08-04        mttr      0.071   8.244341
            ep_Cybersecurity_Operations_20250818             Cybersecurity Operations 2025-08-18 2025-09-15      volume      0.111  22.716668
            ep_Cybersecurity_Operations_20251103             Cybersecurity Operations 2025-11-03 2025-11-10        mttr      0.187   6.453541
            ep_Cybersecurity_Operations_20251117             Cybersecurity Operations 2025-11-17 2025-11-24      volume     

In [31]:
# ── Static candidates ─────────────────────────────────────────────────────────
static_cands = static_candidates.copy()
static_cands["candidate_id"] = (
    "st_" + static_cands["Service"].str.replace(" ", "_")
    + "_" + static_cands["Subcategory"].str.replace(" ", "_")
)
static_cands["candidate_type"] = "persistent"

# ── Cross-reference: does any static candidate's service also appear
#    in temporal candidates? If so, mark as "both".
temporal_services = set(temporal_candidates["Service"].unique())

static_cands["candidate_type"] = np.where(
    static_cands["Service"].isin(temporal_services),
    "both",
    "persistent",
)

static_services = set(static_cands["Service"].unique())
temporal_candidates["candidate_type"] = np.where(
    temporal_candidates["Service"].isin(static_services),
    "both",
    "episodic",
)

# ── Unified candidate list ────────────────────────────────────────────────────
candidate_list = pd.concat([
    temporal_candidates[["candidate_id", "candidate_type", "Service",
                         "start_week", "end_week", "spike_arm",
                         "spike_weeks", "peak_ratio", "total_tickets",
                         "dup_share", "severity"]],
    static_cands[["candidate_id", "candidate_type", "Service",
                  "Subcategory", "volume", "median_MTTR", "impact_score"]],
], ignore_index=True)

print(f"\nCandidate list assembled: {len(candidate_list)} entries")
print(f"  Episodic only:   {(candidate_list['candidate_type'] == 'episodic').sum()}")
print(f"  Persistent only: {(candidate_list['candidate_type'] == 'persistent').sum()}")
print(f"  Both:            {(candidate_list['candidate_type'] == 'both').sum()}")


Candidate list assembled: 46 entries
  Episodic only:   22
  Persistent only: 9
  Both:            15


## Chunk B — Output Summary

| Object | Shape | Description |
|---|---|---|
| `candidate_list` | (candidates × cols) | Unified list with `candidate_id`, `candidate_type` (episodic/persistent/both), `needs_nlp`, `structured_explanation` |
| `static_candidates` | (pairs × 6) | B1 persistent burden: Service, Subcategory, volume, median_MTTR, impact_score, rank |
| `vol_episodes` | (episodes × cols) | Volume spike episodes with severity scores |
| `mttr_episodes` | (episodes × cols) | MTTR spike episodes with severity scores |
| `service_weekly` | augmented | Now carries spike flags, baselines, episode IDs (built on A3 version) |

**Contract with Chunk C:**
- Chunk C iterates over `candidate_list`
- For episodic candidates: filter `df_clean` to `(Service, start_week..end_week)`
- For persistent candidates: filter `df_clean` to `(Service, Subcategory)`
- NLP must not run on any service-week pair not present in `candidate_list`

In [32]:
# ── Verify outputs ─────────────────────────────────────────────────────────────
print("Chunk B outputs:")
for name, obj in [
    ("candidate_list",     candidate_list),
    ("static_candidates",  static_candidates),
    ("vol_episodes",       vol_episodes),
    ("mttr_episodes",      mttr_episodes),
    ("service_weekly",     service_weekly),
]:
    print(f"  {name:25s} {str(obj.shape):>15s}")

print(f"\n  candidate_list columns: {list(candidate_list.columns)}")
print("\nChunk B complete. Ready for Chunk C.")

Chunk B outputs:
  candidate_list                   (46, 15)
  static_candidates                 (17, 6)
  vol_episodes                     (38, 13)
  mttr_episodes                    (64, 13)
  service_weekly                 (3045, 19)

  candidate_list columns: ['candidate_id', 'candidate_type', 'Service', 'start_week', 'end_week', 'spike_arm', 'spike_weeks', 'peak_ratio', 'total_tickets', 'dup_share', 'severity', 'Subcategory', 'volume', 'median_MTTR', 'impact_score']

Chunk B complete. Ready for Chunk C.


---

# Stage 3 - Deep Dive 

NLP clustering, gated by the `candidate_list` from Stage 2.

Architecture:
- **C0** prepares NLP text on *all* tickets (fast, regex-only), but clustering
  only runs on candidates.
- **C1** loads the embedding model once.
- **C1.5** runs structured field analysis for every candidate — this is the
  old B5 logic, repositioned as *analysis* rather than a gate.
- **C2** runs embed → UMAP → HDBSCAN → TF-IDF labeling per candidate.
- **C3** validates cluster quality.
- **C4** synthesizes case studies, combining structured + NLP findings.

Every candidate gets both a structured summary *and* (where text exists) NLP
clusters. The case study narrative then decides which adds signal.

## C0 — NLP Text Preparation

Extracts work notes from the `Comments and Work notes` field, cleans text,
and adds NLP columns to `df_clean`.

Three-level fallback for NLP input:
1. **Work notes (Internal)** from human agents — highest signal
2. **Short description** — fallback when work notes are absent
3. Skip — text too short after cleaning (<15 chars)

Applied to all tickets; clustering in C2 is gated by `candidate_list`.

In [33]:
import re

# ── Timestamp format variants ServiceNow may produce ─────────────────────────
_SEGMENT_RE = re.compile(
    r'(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:\s*(?:AM|PM))?)' 
    r'\s*-\s*'
    r'(?P<author>[^\(]+)'
    r'\((?P<seg_type>[^\)]+)\)'
    r'\n?(?P<content>.*?)'
    r'(?=\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}|\Z)',
    re.DOTALL | re.IGNORECASE
)


def extract_work_notes_signal(text):
    """
    Extract Work notes (Internal) segments from human agents only.
    Discards Additional comments (routing templates, email chains)
    and System-generated work notes.

    Returns (signal_text, extraction_reason) for coverage diagnostics.
    """
    text = str(text)
    if not text.strip() or text == "nan":
        return "", "no_field_content"

    work_note_blocks = []
    any_segment_found = False

    for match in _SEGMENT_RE.finditer(text):
        any_segment_found = True
        author   = match.group("author").strip()
        seg_type = match.group("seg_type").strip()
        content  = match.group("content").strip()
        if "Work notes" in seg_type and author.lower() != "system":
            work_note_blocks.append(content)

    if work_note_blocks:
        return " ".join(work_note_blocks).strip(), "extracted"
    elif any_segment_found:
        return "", "no_work_note_segments"
    else:
        return "", "regex_no_match"


# Apply extraction
MIN_SIGNAL = CONFIG["min_nlp_chars"]

_result = df_clean["Comments and Work notes"].apply(extract_work_notes_signal)
df_clean["work_notes_signal"]    = _result.apply(lambda x: x[0])
df_clean["_extraction_reason"]   = _result.apply(lambda x: x[1])
df_clean["has_work_note_signal"] = df_clean["work_notes_signal"].str.len() >= MIN_SIGNAL

print("Work notes extraction — coverage:")
print(f"  Extracted:            {(df_clean['_extraction_reason'] == 'extracted').mean():.1%}")
print(f"  No work note segs:   {(df_clean['_extraction_reason'] == 'no_work_note_segments').mean():.1%}")
print(f"  Regex no match:      {(df_clean['_extraction_reason'] == 'regex_no_match').mean():.1%}")
print(f"  No field content:    {(df_clean['_extraction_reason'] == 'no_field_content').mean():.1%}")
print(f"  Has signal (≥{MIN_SIGNAL} chars): {df_clean['has_work_note_signal'].mean():.1%}")

# Spot-check regex misses
regex_misses = df_clean[df_clean["_extraction_reason"] == "regex_no_match"]["Comments and Work notes"]
if len(regex_misses) > 0:
    print(f"\nRegex miss sample ({len(regex_misses):,} tickets):")
    print(f"  {str(regex_misses.iloc[0])[:300]!r}")

Work notes extraction — coverage:
  Extracted:            72.2%
  No work note segs:   27.6%
  Regex no match:      0.0%
  No field content:    0.2%
  Has signal (≥15 chars): 71.5%


In [34]:
def clean_text(text):
    """
    Clean extracted work note or short description text for NLP.

    Design note on numbers:
    Blanket digit removal also strips IT-diagnostic tokens such as
    error codes, ticket numbers, and device IDs which are discriminative for
    clustering. Only structurally noisy numeric patterns are removed:
    timestamps, phone numbers, and standalone 1-3 digit filler numbers.
    Longer numeric strings (4+ digits) are retained as potential identifiers.
    """
    text = str(text).lower()
    text = re.sub(r"\[code\].*?\[/code\]", " ", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>",                    " ", text)           # residual HTML
    text = re.sub(r"https?://\S+",               " ", text)           # URLs
    text = re.sub(r"\S+@\S+",                   " ", text)           # emails
    text = re.sub(r"\d{3}[-.\s]?\d{3}[-.\s]?\d{4}", " ", text)  # phone numbers
    text = re.sub(r"\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}", " ", text)  # timestamps
    text = re.sub(r"(?<![\w\d])\d{1,3}(?![\d\w])", " ", text)        # standalone short nums
    text = re.sub(r"&[a-z]+;",                   " ", text)           # HTML entities
    text = re.sub(r"[^a-z0-9\s]",               " ", text)           # non-alphanumeric
    text = re.sub(r"\s+",                         " ", text)
    return text.strip()


# Build NLP input: work note signal where available, short description as fallback
df_clean["nlp_text"] = df_clean.apply(
    lambda row: row["work_notes_signal"]
    if row["has_work_note_signal"]
    else str(row.get("Short description", "")),
    axis=1,
).apply(clean_text)

df_clean["nlp_text_len"] = df_clean["nlp_text"].str.len()

print("NLP text — length distribution:")
print(df_clean["nlp_text_len"].describe().round(0).to_string())
print(f"\nTickets with <{MIN_SIGNAL} chars after cleaning: {(df_clean['nlp_text_len'] < MIN_SIGNAL).mean():.1%}")
print(f"Source: work notes {df_clean['has_work_note_signal'].mean():.1%} / short description {(~df_clean['has_work_note_signal']).mean():.1%}")

NLP text — length distribution:
count    270559.0
mean        281.0
std         569.0
min           0.0
25%          46.0
50%         114.0
75%         331.0
max       30956.0

Tickets with <15 chars after cleaning: 2.3%
Source: work notes 71.5% / short description 28.5%


## C1 — Embedding

Load the sentence transformer once. The `embed_subset` utility is called
per-candidate in C2.

In [35]:
from sentence_transformers import SentenceTransformer

print("Loading sentence transformer model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded: all-MiniLM-L6-v2")


def embed_subset(sub_df, batch_size=256):
    """Embed nlp_text for a subset dataframe. Returns numpy array of embeddings."""
    texts = sub_df["nlp_text"].fillna("").tolist()
    return embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

Loading sentence transformer model...
Model loaded: all-MiniLM-L6-v2


## C1.5 — Structured Field Analysis

For every candidate, summarize what structured fields reveal *before* NLP runs.
This is not a gate — every candidate gets this analysis. The output feeds into
C4 case studies: "structured fields show X; NLP adds Y (or confirms X)."

For **episodic** candidates: compare category/subcategory/assignment-group
distributions in the spike window vs the same service's baseline weeks.

For **persistent** candidates: report the subcategory and top assignment groups.

In [36]:
SCREENING_COLS = ["Category", "Subcategory", "Assignment group"]

def structured_analysis_episodic(row, df_clean, service_weekly):
    """
    Compare structured field distributions in spike vs baseline for an episodic candidate.
    Returns a dict with per-field findings.
    """
    service    = row["Service"]
    start_week = row["start_week"]
    end_week   = row["end_week"]

    spike_mask = (
        (df_clean["Service"] == service)
        & (df_clean["week"] >= start_week)
        & (df_clean["week"] <= end_week)
    )
    spike_tickets = df_clean[spike_mask]

    spike_weeks_svc = set(
        service_weekly[
            (service_weekly["Service"] == service) & service_weekly["any_spike"]
        ]["week"]
    )
    baseline_mask = (
        (df_clean["Service"] == service) & ~df_clean["week"].isin(spike_weeks_svc)
    )
    baseline_tickets = df_clean[baseline_mask]

    findings = {
        "n_spike": len(spike_tickets),
        "n_baseline": len(baseline_tickets),
        "fields": {},
    }

    for col in SCREENING_COLS:
        spike_dist    = spike_tickets[col].value_counts(normalize=True)
        baseline_dist = baseline_tickets[col].value_counts(normalize=True)
        if len(spike_dist) == 0:
            continue

        top_val   = spike_dist.index[0]
        top_share = spike_dist.iloc[0]
        base_share = baseline_dist.get(top_val, 0)
        lift = top_share / base_share if base_share > 0 else float("inf")

        # Build comparison table (top 5 values)
        comp = pd.concat([
            spike_dist.head(5).rename("spike"),
            baseline_dist.rename("baseline"),
        ], axis=1).fillna(0)
        comp["lift"] = (comp["spike"] / comp["baseline"].replace(0, float("nan"))).round(2)
        comp = comp.sort_values("lift", ascending=False).head(5)

        findings["fields"][col] = {
            "top_value": top_val,
            "top_share": round(top_share, 3),
            "baseline_share": round(base_share, 3),
            "lift": round(lift, 2),
            "dominates": top_share >= 0.70 and lift >= 1.5,
            "comparison": comp,
        }

    # Determine if structured fields are sufficient
    any_dominant = any(f["dominates"] for f in findings["fields"].values())
    if any_dominant:
        dominant_fields = [
            f"{col}='{f['top_value']}' ({f['top_share']:.0%}, lift={f['lift']:.1f}x)"
            for col, f in findings["fields"].items() if f["dominates"]
        ]
        findings["structured_sufficient"] = True
        findings["explanation"] = "Structured fields explain spike: " + "; ".join(dominant_fields)
    else:
        findings["structured_sufficient"] = False
        findings["explanation"] = "No single structured field dominates — NLP should add signal"

    return findings


def structured_analysis_persistent(row, df_clean):
    """Summarize structured fields for a persistent (static) candidate."""
    service = row["Service"]
    subcat  = row.get("Subcategory", "")

    mask = (df_clean["Service"] == service) & (df_clean["Subcategory"] == subcat)
    tickets = df_clean[mask]

    GENERIC = {"Other", "General", "Inquiry / Help", "Inquiry/Help",
               "Not Defined", "None", "N/A", "Unknown",
               "Move/Change/Add", "Enhancement"}

    findings = {
        "n_tickets": len(tickets),
        "subcategory": subcat,
        "is_generic": subcat in GENERIC,
    }

    if len(tickets) > 0:
        findings["top_category"] = tickets["Category"].value_counts().index[0]
        findings["top_group"]    = tickets["Assignment group"].value_counts().index[0]
        findings["median_MTTR"]  = round(tickets["MTTR_hours"].median(), 1)

    if findings["is_generic"]:
        findings["structured_sufficient"] = False
        findings["explanation"] = f"Subcategory '{subcat}' is generic — NLP needed to resolve what's inside"
    else:
        findings["structured_sufficient"] = True
        findings["explanation"] = f"Subcategory '{subcat}' is specific — structured fields sufficient"

    return findings


# ── Run structured analysis on all candidates ────────────────────────────────
structured_results = {}

for idx, row in candidate_list.iterrows():
    cid = row["candidate_id"]
    if pd.notna(row.get("start_week")):
        structured_results[cid] = structured_analysis_episodic(row, df_clean, service_weekly)
    else:
        structured_results[cid] = structured_analysis_persistent(row, df_clean)

n_suff = sum(1 for r in structured_results.values() if r["structured_sufficient"])
print(f"Structured analysis complete for {len(structured_results)} candidates")
print(f"  Structured sufficient: {n_suff}")
print(f"  NLP should add signal: {len(structured_results) - n_suff}")

# ── Show structured-sufficient cases in detail ────────────────────────────────
print(f"\n{'='*70}")
print(f"STRUCTURED-SUFFICIENT CANDIDATES ({n_suff})")
print(f"{'='*70}")

for cid, findings in structured_results.items():
    if not findings.get("structured_sufficient"):
        continue

    # Find the candidate row for context
    cand_row = candidate_list[candidate_list["candidate_id"] == cid].iloc[0]
    svc  = cand_row["Service"]
    ctype = cand_row["candidate_type"]

    print(f"\n{'━'*70}")
    print(f"  {svc}  [{ctype}]")

    if "n_spike" in findings:
        # Episodic
        print(f"  Spike window: {cand_row['start_week'].strftime('%Y-%m-%d')} → {cand_row['end_week'].strftime('%Y-%m-%d')}")
        print(f"  Tickets in spike: {findings['n_spike']:,}  |  Baseline pool: {findings['n_baseline']:,}")
        print(f"  {findings['explanation']}")

        # Show the comparison tables for dominant fields
        for col, f in findings["fields"].items():
            tag = " ◀ DOMINANT" if f["dominates"] else ""
            print(f"\n  ── {col}{tag} ──")
            print(f"  Top value: '{f['top_value']}'  spike={f['top_share']:.0%}  baseline={f['baseline_share']:.0%}  lift={f['lift']:.1f}x")
            comp = f["comparison"].copy()
            comp.index.name = col
            comp_str = comp.round(3).to_string()
            for line in comp_str.split("\n"):
                print(f"    {line}")
    else:
        # Persistent
        print(f"  Subcategory: '{findings['subcategory']}'")
        print(f"  Tickets: {findings['n_tickets']:,}")
        if "top_category" in findings:
            print(f"  Top category: {findings['top_category']}")
            print(f"  Top assignment group: {findings['top_group']}")
            print(f"  Median MTTR: {findings['median_MTTR']}h")
        print(f"  {findings['explanation']}")

Structured analysis complete for 46 candidates
  Structured sufficient: 6
  NLP should add signal: 40

STRUCTURED-SUFFICIENT CANDIDATES (6)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Coreimage Engineering Services  [both]
  Spike window: 2025-04-21 → 2025-04-28
  Tickets in spike: 1,349  |  Baseline pool: 2,895
  Structured fields explain spike: Category='Service Request' (96%, lift=1.8x); Subcategory='Other' (91%, lift=10.0x)

  ── Category ◀ DOMINANT ──
  Top value: 'Service Request'  spike=96%  baseline=54%  lift=1.8x
                                      spike  baseline  lift
    Category                                               
    Service Recovery And Recognition  0.001     0.000  2.15
    Service Request                   0.964     0.543  1.77
    Inquiry / Help                    0.013     0.145  0.09
    Incident                          0.022     0.311  0.07

  ── Subcategory ◀ DOMINANT ──
  Top value: 'Other'  spike=91%  baseline=9%  lift

## C2 — NLP Clustering

Runs embed → UMAP → HDBSCAN → TF-IDF labeling per candidate.

Geometry safeguards:
- UMAP `n_components` capped at `min(30, n_samples // 3, n_samples - 2)`
- `n_neighbors` capped at `n_samples - 1`
- Minimum viable sample: `max(2 × min_cluster_size, 40)`

`min_cluster_size` auto-scales to 3% of population (floor = 20) unless overridden.

In [37]:
import umap
import hdbscan
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# ── Domain stop words ─────────────────────────────────────────────────────────
DOMAIN_STOPS = {
    "style", "span", "font", "height", "strong", "width", "code", "nbsp",
    "border", "color", "padding", "margin", "background", "display",
    "text", "align", "com", "med", "umich", "email", "http", "https",
    "www", "html", "href", "class", "div", "table", "notes", "work",
    "additional", "comments", "internal", "service", "hits", "help",
    "thank", "thanks", "please", "hello", "dear", "regards", "sincerely",
    "phone", "pager", "cell", "office", "shift",
}
ALL_STOPS = list(set(ENGLISH_STOP_WORDS) | DOMAIN_STOPS)

# ── Geometry constants ────────────────────────────────────────────────────────
_UMAP_MAX_COMPONENTS = 30
_MIN_VIABLE_SAMPLES  = 40


def _safe_umap_params(n_samples, umap_components=_UMAP_MAX_COMPONENTS):
    """Return (n_components, n_neighbors) safe for this sample size."""
    n_comp  = min(umap_components, n_samples // 3, n_samples - 2)
    n_comp  = max(n_comp, 2)
    n_neigh = min(15, n_samples - 1)
    return n_comp, n_neigh

In [38]:
def run_nlp_pipeline(sub, min_cluster_size=None, cluster_pct=0.03):
    """
    Full NLP pipeline for a filtered ticket subset:
      1. Embed with sentence transformer
      2. Reduce with UMAP (geometry-safe)
      3. Cluster with HDBSCAN
      4. Label clusters with top TF-IDF terms
      5. Return (annotated sub_df, cluster_summary)

    Parameters
    ----------
    sub               : DataFrame — pre-filtered, must have nlp_text, nlp_text_len,
                         has_work_note_signal, MTTR_hours, Number, Category/Subcategory,
                         Assignment group columns
    min_cluster_size  : int or None — override auto-scaling
    cluster_pct       : float — auto-scale min_cluster_size to this fraction of pop

    Returns
    -------
    sub     : DataFrame with cluster_id, cluster_label added
    summary : DataFrame with per-cluster statistics
    Returns (None, None) if too few tickets.
    """
    # Auto-scale min_cluster_size
    effective_mcs = (
        min_cluster_size if min_cluster_size is not None
        else max(20, int(len(sub) * cluster_pct))
    )

    min_viable = max(effective_mcs * 2, _MIN_VIABLE_SAMPLES)
    if len(sub) < min_viable:
        print(f"  Too few tickets ({len(sub)} < {min_viable}) — skipping NLP")
        return None, None

    n_comp, n_neigh = _safe_umap_params(len(sub))

    # ── Embed ────────────────────────────────────────────────────────────
    embeddings = embed_subset(sub)

    # ── UMAP ─────────────────────────────────────────────────────────────
    reducer = umap.UMAP(
        n_components=n_comp, metric="cosine",
        random_state=42, n_neighbors=n_neigh,
    )
    reduced = reducer.fit_transform(embeddings)

    # ── HDBSCAN ──────────────────────────────────────────────────────────
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=effective_mcs,
        metric="euclidean",
        prediction_data=True,
    )
    sub = sub.copy()
    sub["cluster_id"] = clusterer.fit_predict(reduced)

    n_clusters = sub[sub["cluster_id"] >= 0]["cluster_id"].nunique()
    noise_rate = (sub["cluster_id"] == -1).mean()

    # ── TF-IDF labels ────────────────────────────────────────────────────
    tfidf = TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, max_df=0.85,
        max_features=3000, stop_words=ALL_STOPS, sublinear_tf=True,
    )
    X  = tfidf.fit_transform(sub["nlp_text"].fillna(""))
    fn = tfidf.get_feature_names_out()

    cluster_labels = {}
    for cid in sorted(sub["cluster_id"].unique()):
        if cid == -1:
            cluster_labels[-1] = "— noise —"
            continue
        cmask = (sub["cluster_id"] == cid).values
        avg   = X[cmask].mean(axis=0).A1
        top   = avg.argsort()[::-1][:5]
        cluster_labels[cid] = " / ".join(fn[i] for i in top)

    sub["cluster_label"] = sub["cluster_id"].map(cluster_labels)

    # ── Summary table ────────────────────────────────────────────────────
    summary = (
        sub.groupby(["cluster_id", "cluster_label"])
        .agg(
            tickets       =("Number",               "count"),
            median_MTTR   =("MTTR_hours",           "median"),
            work_note_pct =("has_work_note_signal",  "mean"),
            top_subcategory=("Subcategory",  lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else "N/A"),
            top_group     =("Assignment group", lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else "N/A"),
        )
        .reset_index()
        .sort_values("tickets", ascending=False)
    )
    summary["pct_of_total"]  = (summary["tickets"] / len(sub) * 100).round(1)
    summary["work_note_pct"] = (summary["work_note_pct"] * 100).round(1)
    summary["median_MTTR"]   = summary["median_MTTR"].round(1)

    print(f"  Clusters: {n_clusters}  |  Noise: {noise_rate:.1%}  |  min_cluster_size: {effective_mcs}")

    return sub, summary

In [39]:
# ── NLP candidate selection ────────────────────────────────────────────────────
# Not every candidate in candidate_list gets NLP.
# Four gates reduce the ~46 candidates to a workable set:
#
#   1. Severity / impact rank gate:
#      - Episodic: top-K by severity (CONFIG["nlp_top_k_episodic"])
#      - Persistent: top-K by impact_score (CONFIG["nlp_top_k_persistent"])
#
#   2. Ticket floor: need ≥ nlp_min_tickets usable tickets after filtering
#
#   3. Duplicate deflation: subtract duplicate cluster tickets from the
#      usable count. A 200-ticket candidate that's 60% duplicates has
#      only 80 usable tickets → below floor → skip.
#
#   4. Population cap: subsample to nlp_max_tickets for large populations
#      (e.g. Device Support × Other with 7,000+ tickets)

MIN_NLP       = CONFIG["min_nlp_chars"]
MIN_TICKETS   = CONFIG["nlp_min_tickets"]
MAX_TICKETS   = CONFIG["nlp_max_tickets"]
TOP_K_EP      = CONFIG["nlp_top_k_episodic"]
TOP_K_ST      = CONFIG["nlp_top_k_persistent"]
DUP_WARN      = CONFIG["dup_deflation_warn"]

# ── Gate 1: Rank selection ────────────────────────────────────────────────────
# Episodic: top-K by severity
episodic = candidate_list[candidate_list["start_week"].notna()].copy()
episodic_selected = episodic.nlargest(TOP_K_EP, "severity")["candidate_id"].tolist()

# Persistent: top-K by impact_score
persistent = candidate_list[candidate_list["Subcategory"].notna()].copy()
persistent_selected = persistent.nlargest(TOP_K_ST, "impact_score")["candidate_id"].tolist()

nlp_candidate_ids = set(episodic_selected + persistent_selected)

print(f"NLP rank gate: {len(nlp_candidate_ids)} candidates selected")
print(f"  Episodic (top {TOP_K_EP} by severity):    {len(episodic_selected)}")
print(f"  Persistent (top {TOP_K_ST} by impact):    {len(persistent_selected)}")

# ── Gates 2-4 applied per candidate ──────────────────────────────────────────
cluster_results = {}
skipped_reasons = {}

nlp_candidates = candidate_list[candidate_list["candidate_id"].isin(nlp_candidate_ids)]

for idx, row in nlp_candidates.iterrows():
    cid = row["candidate_id"]
    svc = row["Service"]

    # ── Filter df_clean to candidate scope ────────────────────────────────
    if pd.notna(row.get("start_week")):
        scope_mask = (
            (df_clean["Service"] == svc)
            & (df_clean["week"] >= row["start_week"])
            & (df_clean["week"] <= row["end_week"])
            & (df_clean["nlp_text_len"] >= MIN_NLP)
        )
        scope_label = f"weeks {row['start_week'].strftime('%Y-%m-%d')} → {row['end_week'].strftime('%Y-%m-%d')}"
    else:
        subcat = row.get("Subcategory", "")
        scope_mask = (
            (df_clean["Service"] == svc)
            & (df_clean["Subcategory"] == subcat)
            & (df_clean["nlp_text_len"] >= MIN_NLP)
        )
        scope_label = f"subcategory '{subcat}'"

    sub = df_clean[scope_mask].copy()

    # ── Gate 3: Duplicate deflation ───────────────────────────────────────
    n_total = len(sub)
    n_dup   = sub["is_duplicate_cluster"].sum()
    dup_pct = n_dup / n_total if n_total > 0 else 0

    # Remove duplicate cluster tickets before NLP
    sub = sub[~sub["is_duplicate_cluster"]].reset_index(drop=True)
    n_usable = len(sub)

    # ── Gate 2: Ticket floor ──────────────────────────────────────────────
    if n_usable < MIN_TICKETS:
        skipped_reasons[cid] = f"Below ticket floor ({n_usable} usable < {MIN_TICKETS}; {n_dup} duplicates removed)"
        cluster_results[cid] = (None, None)
        print(f"\n[{row['candidate_type']:10s}] {svc} | {scope_label}")
        print(f"  Skipped: {n_usable} usable tickets < {MIN_TICKETS} (removed {n_dup} duplicates)")
        continue

    # ── Gate 4: Population cap ────────────────────────────────────────────
    if n_usable > MAX_TICKETS:
        sub = sub.sample(n=MAX_TICKETS, random_state=42).reset_index(drop=True)
        cap_note = f" (subsampled from {n_usable})"
    else:
        cap_note = ""

    print(f"\n[{row['candidate_type']:10s}] {svc} | {scope_label} | {len(sub)} tickets{cap_note}")
    if dup_pct >= DUP_WARN:
        print(f"  ⚠ {dup_pct:.0%} of original tickets were duplicates (removed)")

    sub_result, summary = run_nlp_pipeline(sub)
    cluster_results[cid] = (sub_result, summary)

    if sub_result is None:
        skipped_reasons[cid] = f"NLP pipeline returned None ({len(sub)} tickets)"

# ── Also record skips for candidates that didn't pass Gate 1 ─────────────────
for idx, row in candidate_list.iterrows():
    cid = row["candidate_id"]
    if cid not in nlp_candidate_ids and cid not in cluster_results:
        cluster_results[cid] = (None, None)
        skipped_reasons[cid] = "Below severity/impact rank threshold for NLP"

# ── Summary ───────────────────────────────────────────────────────────────────
n_clustered = sum(1 for v in cluster_results.values() if v[0] is not None)
n_skipped   = sum(1 for v in cluster_results.values() if v[0] is None)
print(f"\n{'='*60}")
print(f"NLP clustering complete")
print(f"  Clustered: {n_clustered} candidates")
print(f"  Skipped:   {n_skipped} candidates")
if skipped_reasons:
    print(f"\nSkip reasons:")
    for cid, reason in sorted(skipped_reasons.items()):
        print(f"  {cid}: {reason}")

NLP rank gate: 20 candidates selected
  Episodic (top 10 by severity):    10
  Persistent (top 10 by impact):    10

[both      ] Coreimage Engineering Services | weeks 2025-04-21 → 2025-04-28 | 1282 tickets


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

/Users/muyu/anaconda3/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


  Clusters: 4  |  Noise: 6.0%  |  min_cluster_size: 38

[episodic  ] Cornerstone Performance Management | weeks 2025-05-05 → 2025-06-30 | 866 tickets


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

/Users/muyu/anaconda3/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Clusters: 2  |  Noise: 49.0%  |  min_cluster_size: 20

[both      ] Database Services | subcategory 'Move/Change/Add' | 404 tickets
  ⚠ 53% of original tickets were duplicates (removed)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/muyu/anaconda3/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Clusters: 2  |  Noise: 0.0%  |  min_cluster_size: 20

NLP clustering complete
  Clustered: 19 candidates
  Skipped:   27 candidates

Skip reasons:
  ep_Cornerstone_Performance_Management_20250728: Below severity/impact rank threshold for NLP
  ep_Cybersecurity_Operations_20251103: Below severity/impact rank threshold for NLP
  ep_Cybersecurity_Operations_20251117: Below severity/impact rank threshold for NLP
  ep_Database_Services_20250303: Below severity/impact rank threshold for NLP
  ep_Database_Services_20250929: Below severity/impact rank threshold for NLP
  ep_Database_Services_20251110: Below severity/impact rank threshold for NLP
  ep_Learning_Delivery_Services_20251208: Below severity/impact rank threshold for NLP
  ep_Linux_Hosting_Services_20250203: Below severity/impact rank threshold for NLP
  ep_Linux_Hosting_Services_20250310: Below severity/impact rank threshold for NLP
  ep_Linux_Hosting_Services_20250825: Below severity/impact rank threshold for NLP
  ep_Michart_Pha

## C3 — Cluster Validation

For each successfully clustered candidate, three coherence checks:
1. **Category alignment** — do clusters correspond to distinct incident types?
2. **MTTR consistency** — do clusters have meaningfully different resolution times?
3. **Assignment group concentration** — do clusters route to different teams?

High coherence on any axis validates that NLP found real structure, not noise.

In [40]:
def validate_candidate(cid, sub, summary):
    """
    Run validation checks on a clustered candidate.
    Returns a validation dict for the case study.
    """
    if sub is None or summary is None:
        return None

    non_noise = sub[sub["cluster_id"] != -1]
    if len(non_noise) == 0:
        return {"status": "all_noise", "cid": cid}

    validation = {"cid": cid, "status": "ok"}

    # 1. Subcategory alignment — do clusters separate different subcategories?
    subcat_cross = pd.crosstab(
        non_noise["cluster_id"], non_noise["Subcategory"], normalize="index"
    )
    # Concentration: for each cluster, how dominant is the top subcategory?
    subcat_concentration = subcat_cross.max(axis=1).mean()
    validation["subcat_concentration"] = round(subcat_concentration, 3)

    # 2. MTTR spread — coefficient of variation across cluster medians
    cluster_mttrs = non_noise.groupby("cluster_id")["MTTR_hours"].median()
    if len(cluster_mttrs) > 1 and cluster_mttrs.mean() > 0:
        mttr_cv = cluster_mttrs.std() / cluster_mttrs.mean()
    else:
        mttr_cv = 0.0
    validation["mttr_cv"] = round(mttr_cv, 3)

    # 3. Assignment group alignment
    group_cross = pd.crosstab(
        non_noise["cluster_id"], non_noise["Assignment group"], normalize="index"
    )
    group_concentration = group_cross.max(axis=1).mean()
    validation["group_concentration"] = round(group_concentration, 3)

    # Overall: NLP adds signal if clusters differ on at least one axis
    validation["nlp_adds_signal"] = (
        subcat_concentration >= 0.6    # clusters correspond to distinct subcategories
        or mttr_cv >= 0.5              # clusters have different resolution times
        or group_concentration >= 0.6  # clusters route to different teams
    )

    return validation


# ── Run validation on all clustered candidates ───────────────────────────────
validation_results = {}

for cid, (sub, summary) in cluster_results.items():
    v = validate_candidate(cid, sub, summary)
    if v is not None:
        validation_results[cid] = v

n_signal = sum(1 for v in validation_results.values() if v.get("nlp_adds_signal"))
n_total  = sum(1 for v in validation_results.values() if v.get("status") == "ok")

print(f"Validation complete: {n_total} clustered candidates")
print(f"  NLP adds signal:   {n_signal}")
print(f"  NLP weak/confirm:  {n_total - n_signal}")
print()

# Detailed table
val_rows = []
for cid, v in validation_results.items():
    if v["status"] != "ok":
        continue
    val_rows.append({
        "candidate_id": cid,
        "subcat_conc": v["subcat_concentration"],
        "mttr_cv": v["mttr_cv"],
        "group_conc": v["group_concentration"],
        "nlp_adds_signal": v["nlp_adds_signal"],
    })

if val_rows:
    val_df = pd.DataFrame(val_rows).sort_values("nlp_adds_signal", ascending=False)
    print(val_df.to_string(index=False))

Validation complete: 18 clustered candidates
  NLP adds signal:   18
  NLP weak/confirm:  0

                                    candidate_id  subcat_conc  mttr_cv  group_conc  nlp_adds_signal
      ep_Coreimage_Engineering_Services_20250421        0.980    0.531       0.329             True
  ep_Cornerstone_Performance_Management_20250505        0.848    1.026       0.691             True
                     st_Reporting_Inquiry_/_Help        1.000    1.026       0.328             True
      st_Device_Support_Services_System_Degraded        1.000    0.803       0.350             True
 st_Pharmacy_Management_Services_Move/Change/Add        1.000    0.612       0.764             True
      st_Device_Support_Services_Move/Change/Add        1.000    0.880       0.288             True
  st_Continuing_Medical_Education_Inquiry_/_Help        1.000    0.638       0.631             True
                st_Device_Support_Services_Other        1.000    0.672       0.195             True
       

## C4 — Case Study Synthesis

For each candidate, produce a structured narrative combining:
1. **What happened** — the spike or persistent burden pattern
2. **What structured fields show** — from C1.5 analysis
3. **What NLP adds** — from C2 clusters + C3 validation

Three outcomes:
- *"Structured fields explain this; NLP confirms"* — no additional insight from text
- *"Structured fields are generic; NLP resolves the content"* — text adds real signal
- *"Insufficient text for NLP"* — structured fields are all we have

In [41]:
def build_case_study(row, structured, cluster_result, validation):
    """
    Build a case study dict for one candidate.

    Parameters
    ----------
    row         : candidate_list row
    structured  : dict from structured_analysis (C1.5)
    cluster_result : (sub_df, summary) or (None, None) from C2
    validation  : dict from C3 or None
    """
    cid   = row["candidate_id"]
    svc   = row["Service"]
    ctype = row["candidate_type"]
    sub, summary = cluster_result

    case = {
        "candidate_id": cid,
        "service": svc,
        "candidate_type": ctype,
    }

    # ── What happened ────────────────────────────────────────────────────
    if pd.notna(row.get("start_week")):
        case["scope"] = f"{row['start_week'].strftime('%Y-%m-%d')} → {row['end_week'].strftime('%Y-%m-%d')}"
        case["spike_arm"]     = row.get("spike_arm", "")
        case["peak_ratio"]    = row.get("peak_ratio", None)
        case["total_tickets"] = row.get("total_tickets", None)
        case["severity"]      = row.get("severity", None)
    else:
        case["scope"]        = f"subcategory '{row.get('Subcategory', '')}'"
        case["volume"]       = row.get("volume", None)
        case["median_MTTR"]  = row.get("median_MTTR", None)
        case["impact_score"] = row.get("impact_score", None)

    # ── What structured fields show ──────────────────────────────────────
    case["structured_sufficient"] = structured.get("structured_sufficient", False)
    case["structured_explanation"] = structured.get("explanation", "")

    # ── What NLP adds ────────────────────────────────────────────────────
    if sub is not None and summary is not None:
        n_clusters = summary[summary["cluster_id"] >= 0].shape[0]
        noise_pct  = summary.loc[summary["cluster_id"] == -1, "pct_of_total"].sum() if -1 in summary["cluster_id"].values else 0

        case["nlp_ran"] = True
        case["n_clusters"] = n_clusters
        case["noise_pct"]  = round(noise_pct, 1)
        case["cluster_labels"] = summary[summary["cluster_id"] >= 0][
            ["cluster_id", "cluster_label", "tickets", "pct_of_total", "median_MTTR"]
        ].to_dict("records")

        if validation and validation.get("nlp_adds_signal"):
            case["nlp_verdict"] = "NLP adds signal"
            case["nlp_detail"] = (
                f"subcat_conc={validation['subcat_concentration']}, "
                f"mttr_cv={validation['mttr_cv']}, "
                f"group_conc={validation['group_concentration']}"
            )
        else:
            case["nlp_verdict"] = "NLP confirms structured findings"
            case["nlp_detail"] = "Clusters do not separate on subcategory, MTTR, or assignment group"
    else:
        case["nlp_ran"] = False
        case["nlp_verdict"] = "Insufficient text for NLP"
        case["nlp_detail"] = skipped_reasons.get(cid, "Unknown reason")

    return case


# ── Build all case studies ────────────────────────────────────────────────────
case_studies = []

for idx, row in candidate_list.iterrows():
    cid = row["candidate_id"]
    structured = structured_results.get(cid, {})
    cluster_result = cluster_results.get(cid, (None, None))
    validation = validation_results.get(cid, None)

    case = build_case_study(row, structured, cluster_result, validation)
    case_studies.append(case)

print(f"Case studies generated: {len(case_studies)}")
print(f"  NLP adds signal:               {sum(1 for c in case_studies if c.get('nlp_verdict') == 'NLP adds signal')}")
print(f"  NLP confirms structured:       {sum(1 for c in case_studies if c.get('nlp_verdict') == 'NLP confirms structured findings')}")
print(f"  Insufficient text:             {sum(1 for c in case_studies if c.get('nlp_verdict') == 'Insufficient text for NLP')}")

Case studies generated: 46
  NLP adds signal:               18
  NLP confirms structured:       1
  Insufficient text:             27


In [42]:
# ── Print case studies ─────────────────────────────────────────────────────────

def print_case_study(case):
    """Pretty-print a single case study."""
    print(f"\n{'━'*70}")
    print(f"  {case['service']}  [{case['candidate_type']}]")
    print(f"  Scope: {case['scope']}")
    print(f"{'━'*70}")

    # What happened
    if "peak_ratio" in case and case["peak_ratio"] is not None:
        print(f"  Spike: {case['spike_arm']}  |  peak ratio: {case['peak_ratio']:.1f}x"
              f"  |  tickets: {case['total_tickets']:,.0f}  |  severity: {case['severity']:.1f}")
    elif "impact_score" in case and case["impact_score"] is not None:
        print(f"  Persistent: {case['volume']:,.0f} tickets"
              f"  |  median MTTR: {case['median_MTTR']:.1f}h"
              f"  |  impact: {case['impact_score']:,.0f}")

    # Structured fields
    print(f"\n  Structured: {'✓ sufficient' if case['structured_sufficient'] else '✗ insufficient'}")
    print(f"  {case['structured_explanation']}")

    # NLP
    print(f"\n  NLP: {case['nlp_verdict']}")
    if case["nlp_ran"]:
        print(f"  {case['n_clusters']} clusters, {case['noise_pct']:.0f}% noise")
        print(f"  Validation: {case['nlp_detail']}")
        for cl in case.get("cluster_labels", []):
            print(f"    C{cl['cluster_id']:2d}  ({cl['pct_of_total']:5.1f}%, MTTR {cl['median_MTTR']:.1f}h)  {cl['cluster_label']}")
    else:
        print(f"  {case['nlp_detail']}")


# Print all case studies, sorted: NLP-adds-signal first, then confirms, then insufficient
priority = {"NLP adds signal": 0, "NLP confirms structured findings": 1, "Insufficient text for NLP": 2}
sorted_cases = sorted(case_studies, key=lambda c: (priority.get(c["nlp_verdict"], 9), -c.get("severity", 0) if c.get("severity") else 0))

for case in sorted_cases:
    print_case_study(case)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Michart Pharmacy (Willow Ambulatory)  [episodic]
  Scope: 2025-05-12 → 2025-06-09
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Spike: volume  |  peak ratio: 253.3x  |  tickets: 595  |  severity: 102.1

  Structured: ✗ insufficient
  No single structured field dominates — NLP should add signal

  NLP: NLP adds signal
  2 clusters, 0% noise
  Validation: subcat_conc=0.667, mttr_cv=0.936, group_conc=0.948
    C 0  ( 54.3%, MTTR 26.1h)  support request / ambulatory support / willow ambulatory / willow / support
    C 1  ( 45.7%, MTTR 128.1h)  wam / issue / use / ndc / michart

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Cornerstone Performance Management  [episodic]
  Scope: 2025-05-05 → 2025-06-30
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Spike: volume  |  peak ratio: 4.5x  |  tickets: 982  |  severity: 33.3

  Structured: ✗ insu

# Output

## Pipeline Output Summary

| Stage | Key Outputs |
|---|---|
| **A (Foundation)** | `df_clean`, `service_weekly`, `service_counts`, `weekly_global`, `caller_concentration_df`, `duplicate_clusters_df` |
| **B (Ranking)** | `candidate_list`, `static_candidates`, `vol_episodes`, `mttr_episodes` |
| **C (Deep Dive)** | `structured_results`, `cluster_results`, `validation_results`, `case_studies` |

All NLP was scoped to candidates from Stage 2. No service-week pair outside
`candidate_list` was clustered.

In [43]:
# ── Final output verification ─────────────────────────────────────────────────
print("Pipeline complete.\n")

print("Stage A outputs:")
for name, obj in [("df_clean", df_clean), ("service_weekly", service_weekly),
                   ("service_counts", service_counts), ("weekly_global", weekly_global)]:
    print(f"  {name:30s} {str(obj.shape)}")

print("\nStage B outputs:")
for name, obj in [("candidate_list", candidate_list), ("static_candidates", static_candidates),
                   ("vol_episodes", vol_episodes), ("mttr_episodes", mttr_episodes)]:
    print(f"  {name:30s} {str(obj.shape)}")

print("\nStage C outputs:")
print(f"  {'structured_results':30s} {len(structured_results)} candidates analysed")
print(f"  {'cluster_results':30s} {sum(1 for v in cluster_results.values() if v[0] is not None)} clustered / {len(cluster_results)} total")
print(f"  {'validation_results':30s} {len(validation_results)} validated")
print(f"  {'case_studies':30s} {len(case_studies)} case studies")

print(f"\nCase study verdicts:")
for verdict in ["NLP adds signal", "NLP confirms structured findings", "Insufficient text for NLP"]:
    n = sum(1 for c in case_studies if c["nlp_verdict"] == verdict)
    print(f"  {verdict:40s} {n}")

Pipeline complete.

Stage A outputs:
  df_clean                       (270559, 39)
  service_weekly                 (3045, 19)
  service_counts                 (1002, 3)
  weekly_global                  (51, 4)

Stage B outputs:
  candidate_list                 (46, 15)
  static_candidates              (17, 6)
  vol_episodes                   (38, 13)
  mttr_episodes                  (64, 13)

Stage C outputs:
  structured_results             46 candidates analysed
  cluster_results                19 clustered / 46 total
  validation_results             19 validated
  case_studies                   46 case studies

Case study verdicts:
  NLP adds signal                          18
  NLP confirms structured findings         1
  Insufficient text for NLP                27


In [44]:
# ── Pipeline exports ──────────────────────────────────────────────────────────
import os
OUT = "pipeline_outputs"
os.makedirs(OUT, exist_ok=True)

# 1. Candidate list (what Stage 2 produced)
candidate_list.to_csv(f"{OUT}/candidate_list.csv", index=False)

# 2. NLP cluster summaries (only candidates where NLP ran)
nlp_summaries = []
for cid, (sub, summary) in cluster_results.items():
    if summary is not None:
        s = summary.copy()
        s["candidate_id"] = cid
        nlp_summaries.append(s)

if nlp_summaries:
    pd.concat(nlp_summaries, ignore_index=True).to_csv(
        f"{OUT}/nlp_cluster_summaries.csv", index=False
    )

# 3. Case studies as JSON (for programmatic access)
import json
with open(f"{OUT}/case_studies.json", "w") as f:
    json.dump(case_studies, f, indent=2, default=str)

# 4. Structured analysis results
structured_rows = []
for cid, findings in structured_results.items():
    row = {"candidate_id": cid}
    row["structured_sufficient"] = findings.get("structured_sufficient")
    row["explanation"] = findings.get("explanation")
    if "n_spike" in findings:
        row["n_spike"] = findings["n_spike"]
        row["n_baseline"] = findings["n_baseline"]
    if "n_tickets" in findings:
        row["n_tickets"] = findings["n_tickets"]
    structured_rows.append(row)

pd.DataFrame(structured_rows).to_csv(f"{OUT}/structured_analysis.csv", index=False)

print(f"Exported to {OUT}/:")
for f in sorted(os.listdir(OUT)):
    print(f"  {f}")

Exported to pipeline_outputs/:
  candidate_list.csv
  case_studies.json
  nlp_cluster_summaries.csv
  structured_analysis.csv
  summary.html


## Pipeline Dashboard

Generates a self-contained HTML summary at `pipeline_outputs/summary.html`.
Embeds Altair charts inline and renders candidate tables from pipeline objects.

In [45]:
# ── HTML Summary Dashboard ────────────────────────────────────────────────────
# Generates a self-contained HTML file with embedded Altair charts and
# candidate tables from pipeline outputs. No external dependencies.

import os
OUT = "pipeline_outputs"
os.makedirs(OUT, exist_ok=True)


# ── Persistent burden table ───────────────────────────────────────────────────
_static = candidate_list[candidate_list["Subcategory"].notna()].copy()
_static = _static.sort_values("impact_score", ascending=False)
_static["median_MTTR"] = _static["median_MTTR"].round(1)
_static["impact_score"] = _static["impact_score"].round(0).astype(int)
static_html = (
    _static[["Service", "Subcategory", "candidate_type", "volume", "median_MTTR", "impact_score"]]
    .rename(columns={"candidate_type": "Type", "volume": "Tickets",
                     "median_MTTR": "MTTR (h)", "impact_score": "Impact"})
    .to_html(index=False, classes="data-table", border=0)
)

# ── Episodic disruptions table ────────────────────────────────────────────────
_episodic = candidate_list[candidate_list["start_week"].notna()].copy()
_episodic = _episodic.sort_values("severity", ascending=False)
_episodic["start_week"] = _episodic["start_week"].dt.strftime("%Y-%m-%d")
_episodic["end_week"] = _episodic["end_week"].dt.strftime("%Y-%m-%d")
_episodic["peak_ratio"] = _episodic["peak_ratio"].round(1)
_episodic["dup_share"] = (_episodic["dup_share"] * 100).round(0).astype(int).astype(str) + "%"
episodic_html = (
    _episodic[["Service", "candidate_type", "start_week", "end_week",
               "spike_arm", "peak_ratio", "total_tickets", "dup_share"]]
    .rename(columns={"candidate_type": "Type", "start_week": "Start",
                     "end_week": "End", "spike_arm": "Spike",
                     "peak_ratio": "Peak×", "total_tickets": "Tickets",
                     "dup_share": "Dup%"})
    .to_html(index=False, classes="data-table", border=0)
)

# ── NLP results table (from exported CSV or in-memory) ────────────────────────
nlp_rows = []
for cid, (sub, summary) in cluster_results.items():
    if summary is None:
        continue
    cand = candidate_list[candidate_list["candidate_id"] == cid].iloc[0]
    non_noise = summary[summary["cluster_id"] >= 0]
    n_clusters = len(non_noise)
    noise_pct = summary.loc[summary["cluster_id"] == -1, "pct_of_total"].sum() if -1 in summary["cluster_id"].values else 0

    # Top cluster label
    if len(non_noise) > 0:
        top_label = non_noise.iloc[0]["cluster_label"]
    else:
        top_label = "—"

    nlp_rows.append({
        "Service": cand["Service"],
        "Scope": (f"{cand['start_week'].strftime('%Y-%m-%d')} → {cand['end_week'].strftime('%Y-%m-%d')}"
                  if pd.notna(cand.get("start_week"))
                  else f"'{cand.get('Subcategory', '')}'"),
        "Clusters": n_clusters,
        "Noise%": f"{noise_pct:.0f}%",
        "Top cluster": top_label[:60],
    })

if nlp_rows:
    nlp_html = pd.DataFrame(nlp_rows).to_html(index=False, classes="data-table", border=0)
else:
    nlp_html = "<p>No NLP results available.</p>"


# ── Use charts from A5 ───────────────────────────────────────────────────────
chart_trend_html = chart1.to_html(fullhtml=False, output_div="vis_trend")
chart_heatmap_html = chart3.to_html(fullhtml=False, output_div="vis_heatmap")
chart_scatter_html = chart5.to_html(fullhtml=False, output_div="vis_scatter")

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>HITS Incident Analysis — Pipeline Summary</title>
<style>
  body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    max-width: 900px; margin: 40px auto; padding: 0 20px;
    color: #333; background: #fafafa; line-height: 1.5;
  }}
  h1 {{ font-size: 1.6rem; border-bottom: 2px solid #333; padding-bottom: 8px; }}
  h2 {{ font-size: 1.2rem; color: #555; margin-top: 2.5rem; }}
  h3 {{ font-size: 1.0rem; color: #666; margin-top: 1.5rem; }}
  p.caption {{ font-size: 0.85rem; color: #666; margin-top: 4px; }}
  .data-table {{
    width: 100%; border-collapse: collapse; font-size: 0.82rem; margin: 12px 0;
  }}
  .data-table th {{
    background: #f0f0f0; text-align: left; padding: 6px 10px;
    border-bottom: 2px solid #ccc; font-weight: 600;
  }}
  .data-table td {{
    padding: 5px 10px; border-bottom: 1px solid #e8e8e8;
  }}
  .data-table tr:hover td {{ background: #f5f5f5; }}
  .chart-container {{ margin: 16px 0; }}
  .stats {{
    display: flex; gap: 24px; flex-wrap: wrap; margin: 16px 0;
  }}
  .stat-box {{
    background: white; border: 1px solid #ddd; border-radius: 6px;
    padding: 12px 20px; min-width: 140px;
  }}
  .stat-box .number {{ font-size: 1.5rem; font-weight: 700; color: #333; }}
  .stat-box .label {{ font-size: 0.8rem; color: #888; }}
</style>
</head>
<body>

<h1>Michigan Medicine HITS — Pipeline Summary</h1>

<div class="stats">
  <div class="stat-box">
    <div class="number">{len(df_clean):,}</div>
    <div class="label">Total tickets</div>
  </div>
  <div class="stat-box">
    <div class="number">{df_clean['week'].nunique()}</div>
    <div class="label">Weeks analysed</div>
  </div>
  <div class="stat-box">
    <div class="number">{df_clean['Service'].nunique():,}</div>
    <div class="label">Services</div>
  </div>
  <div class="stat-box">
    <div class="number">{len(candidate_list)}</div>
    <div class="label">Candidates identified</div>
  </div>
  <div class="stat-box">
    <div class="number">{sum(1 for v in cluster_results.values() if v[0] is not None)}</div>
    <div class="label">NLP clustered</div>
  </div>
</div>

<h2>1. Baseline</h2>
<div class="chart-container">{chart_trend_html}</div>
<p class="caption">Demand is stable (CV ≈ {cv:.2f}). Rolling average smooths week-to-week variation.</p>

<div class="chart-container">{chart_heatmap_html}</div>
<p class="caption">Ticket volume by service × subcategory for the top 10 services. Most demand concentrates in a few cells.</p>

<h2>2. Problem Candidates</h2>
<div class="chart-container">{chart_scatter_html}</div>
<p class="caption">Each point is a service–subcategory pair. Top-right = high volume and slow resolution. Top 10 by impact labeled.</p>

<h3>Persistent Burden ({len(_static)} candidates)</h3>
{static_html}
<p class="caption">Impact = Volume × median MTTR</p>

<h3>Episodic Disruptions ({len(_episodic)} candidates)</h3>
{episodic_html}
<p class="caption">Dup%: coarse ticket repetition rate (same caller / service / subcategory within 72hrs)</p>

<h2>3. NLP Cluster Results ({sum(1 for v in cluster_results.values() if v[0] is not None)} candidates clustered)</h2>
{nlp_html}
<p class="caption">Sentence embedding → UMAP → HDBSCAN on agent work notes. Top cluster label shows discriminative TF-IDF terms.</p>

</body>
</html>"""

# ── Write ─────────────────────────────────────────────────────────────────────
out_path = f"{OUT}/summary.html"
with open(out_path, "w") as f:
    f.write(html)

print(f"Dashboard written to {out_path}")
print(f"  File size: {os.path.getsize(out_path) / 1024:.0f} KB")


Dashboard written to pipeline_outputs/summary.html
  File size: 174 KB
